In [ ]:
from dataclasses import asdict, dataclass
from pathlib import Path
import copy
import hashlib
import json
import platform
import shutil
import tempfile

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
from IPython.display import HTML, Image, display

In [ ]:
ANALYSIS_FOLDER = next((candidate_folder for candidate_folder in [Path.cwd(), *Path.cwd().parents]
             if (candidate_folder / "collection_run.json").is_file()
             or (candidate_folder / "inputs/daily/input_manifest.json").is_file()), None)
if ANALYSIS_FOLDER is None:
    raise FileNotFoundError("Run the collection notebook first, then open the analysis notebook copied into its output folder; an original prepared-input folder can also be used.")
collection_record = ANALYSIS_FOLDER / "collection_run.json"
collection_details = json.loads(collection_record.read_text()) if collection_record.is_file() else {"profile": "study_replay"}
if collection_details.get("profile") not in {"study_replay", "new_collection"}:
    raise ValueError("Choose an original study archive or a prepared new collection.")
reproduce_study = collection_details["profile"] == "study_replay"
if not reproduce_study:
    collection_manifest = ANALYSIS_FOLDER / "prepared/new_collection_manifest.json"
    if hashlib.sha256(collection_manifest.read_bytes()).hexdigest() != collection_details["prepared_manifest_sha256"]:
        raise ValueError("The prepared collection manifest has changed.")
    prepared_collection = json.loads(collection_manifest.read_text())
    for home_entry in prepared_collection["homes"]:
        relative_folder = Path(home_entry["relative_folder"])
        home_folder = (ANALYSIS_FOLDER / relative_folder).resolve()
        if relative_folder.is_absolute() or ".." in relative_folder.parts or ANALYSIS_FOLDER.resolve() not in home_folder.parents:
            raise ValueError("Prepared household inputs must remain inside the collection folder.")
        if hashlib.sha256((home_folder / "input_manifest.json").read_bytes()).hexdigest() != home_entry["manifest_sha256"]:
            raise ValueError("A household input manifest has changed.")
INPUTS_FOLDER = ANALYSIS_FOLDER / "inputs"
FIGURES_FOLDER = ANALYSIS_FOLDER / "figures"
FIGURES = FIGURES_FOLDER
OBSERVATION_INPUTS = INPUTS_FOLDER / "observations"
OBSERVATION_OUTPUTS = ANALYSIS_FOLDER / "outputs/observations"
DAILY_OUTPUTS = ANALYSIS_FOLDER / "outputs/daily"
WEATHER_OUTPUTS = ANALYSIS_FOLDER / "outputs/comparison"
SENSOR_INPUTS = INPUTS_FOLDER / "sensor_value"
SENSOR_OUTPUTS = ANALYSIS_FOLDER / "outputs/sensor_value"
for folder in ([FIGURES_FOLDER, OBSERVATION_OUTPUTS, DAILY_OUTPUTS, WEATHER_OUTPUTS, SENSOR_OUTPUTS, ANALYSIS_FOLDER / "checks"] if reproduce_study else []):
    folder.mkdir(parents=True, exist_ok=True)
OBSERVATION_DARK, OBSERVATION_MID, TEXT_COLOUR = "#404040", "#777777", "#303030"
MODEL_DARK, MODEL_MID, MODEL_GREY = "#404040", "#777777", "#555555"
WEATHER_SOURCES = {"Local outdoor": "local_outdoor_C", "ERA5": "era5_outdoor_C"}
LATER_PERIOD_START = pd.Timestamp("2026-04-04 15:30", tz="UTC")
SENSOR_MEASURES = {"flow_mean_C": "Average flow temperature", "gap_mean_C": "Average flow-return gap"}
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 10)

In [ ]:
def load_observations():
    coverage = pd.read_csv(OBSERVATION_INPUTS / "monthly_availability.csv")
    pipes = {
        channel: pd.read_csv(OBSERVATION_INPUTS / f"h2_{channel}_selected.csv")
        for channel in ("flow", "return")
    }
    for pipe_observations in pipes.values():
        pipe_observations["timestamp"] = pd.to_datetime(pipe_observations["timestamp"], utc=True)
    gas = pd.read_csv(OBSERVATION_INPUTS / "h2_gas_selected.csv")
    for column in ("interval_start", "interval_end"):
        gas[column] = pd.to_datetime(gas[column], utc=True)
    return coverage, pipes, gas

In [ ]:
def summarise_availability(coverage):
    count_columns = ["expected_intervals", "recorded_intervals", "excluded_known_frozen_intervals", "retained_intervals"]
    totals = coverage.groupby(["home", "channel"], sort=False)[count_columns].sum().reset_index()
    totals["coverage_percent"] = 100 * totals["retained_intervals"] / totals["expected_intervals"]
    return totals

In [ ]:
def export_observation_tables(coverage, pipes, gas):
    OBSERVATION_OUTPUTS.mkdir(parents=True, exist_ok=True)
    coverage.to_csv(OBSERVATION_OUTPUTS / "F01_monthly_availability.csv", index=False)
    totals = summarise_availability(coverage)
    totals.to_csv(OBSERVATION_OUTPUTS / "availability_totals.csv", index=False)
    for channel, observations in pipes.items():
        observations.to_csv(OBSERVATION_OUTPUTS / f"F02_{channel}_observations.csv", index=False)
    gas.to_csv(OBSERVATION_OUTPUTS / "F02_gas_intervals.csv", index=False)
    return totals

In [ ]:
if reproduce_study:
    monthly_coverage, pipe_observations, gas_intervals = load_observations()
    availability = export_observation_tables(monthly_coverage, pipe_observations, gas_intervals)
    display(HTML(availability[["home", "channel", "retained_intervals", "expected_intervals", "coverage_percent"]].round(1).to_html(index=False)))

In [ ]:
def set_observation_chart_style():
    plt.rcParams.update({
        "font.family": "DejaVu Sans", "font.size": 9,
        "text.color": TEXT_COLOUR, "axes.labelcolor": TEXT_COLOUR,
        "xtick.color": TEXT_COLOUR, "ytick.color": TEXT_COLOUR,
        "axes.edgecolor": "#A0A0A0", "svg.hashsalt": "observe-explain-check",
        "axes.spines.top": False, "axes.spines.right": False,
    })

In [ ]:
def save_observation_figure(figure, name):
    FIGURES.mkdir(parents=True, exist_ok=True)
    figure.savefig(FIGURES / f"{name}.png", dpi=240, facecolor="white")
    figure.savefig(FIGURES / f"{name}.svg", facecolor="white", metadata={"Date": None})
    return figure

In [ ]:
def plot_availability(coverage):
    set_observation_chart_style()
    channels = ["Flow pipe", "Return pipe", "Cylinder outlet", "Gas input", "Weather input"]
    months = sorted(coverage["month"].unique())
    palette = LinearSegmentedColormap.from_list("availability", ["#F7F7F7", OBSERVATION_DARK])
    figure, axes = plt.subplots(2, 1, figsize=(7.2, 3.8))
    figure.subplots_adjust(left=.205, right=.98, top=.82, bottom=.075, hspace=.54)
    figure.text(.02, .95, "Input availability (%)", fontsize=13, weight="bold")
    for axis, home in zip(axes, ("H1", "H2")):
        monthly_availability = coverage.loc[coverage["home"].eq(home)].pivot(index="channel", columns="month", values="coverage_percent")
        monthly_availability = monthly_availability.reindex(index=channels, columns=months)
        axis.imshow(monthly_availability.to_numpy(), vmin=0, vmax=100, cmap=palette, aspect="auto")
        axis.set_title(home, loc="left", fontsize=10, weight="bold", pad=6)
        axis.set_yticks(range(len(channels)), channels)
        axis.set_xticks(range(len(months)), [pd.Timestamp(month).strftime("%b") for month in months])
        axis.tick_params(length=0, labelsize=8.5)
        axis.set_xticks(np.arange(-.5, len(months), 1), minor=True)
        axis.set_yticks(np.arange(-.5, len(channels), 1), minor=True)
        axis.grid(which="minor", color="white", linewidth=1.5)
        axis.tick_params(which="minor", bottom=False, left=False)
        for spine in axis.spines.values():
            spine.set_visible(False)
        for row, column in np.ndindex(monthly_availability.shape):
            coverage_percent = monthly_availability.iloc[row, column]
            if pd.isna(coverage_percent):
                axis.add_patch(Rectangle((column - .5, row - .5), 1, 1, facecolor="white", hatch="////", edgecolor="#777777", linewidth=0))
                label, colour = "–", TEXT_COLOUR
            else:
                label, colour = f"{coverage_percent:.0f}", "white" if coverage_percent >= 71 else "#000000"
            axis.text(column, row, label, ha="center", va="center", fontsize=8.3, color=colour)
    return save_observation_figure(figure, "F01_availability")

In [ ]:
if reproduce_study:
    availability_figure = plot_availability(monthly_coverage)
    plt.close(availability_figure)
    display(Image(filename=str(FIGURES_FOLDER / "F01_availability.png")))

In [ ]:
REFERENCE_TEMPERATURES = np.arange(10.0, 24.01, 0.25)
MINIMUM_WEATHER_HOURS = 20
MINIMUM_TRAINING_DAYS = 60
MINIMUM_ASSESSMENT_DAYS = 10
COMPARISON_METHODS = ("Fitted balance temperature", "Training-period mean", "Fixed 15.5 C baseline")

In [ ]:
def calculate_degree_days(hourly_temperature):
    temperature = hourly_temperature.dropna()
    dates = pd.Index(temperature.index.date, name="date")
    temperature_shortfalls = np.maximum(0.0, REFERENCE_TEMPERATURES[None, :] - temperature.to_numpy()[:, None])
    daily_degree_days = pd.DataFrame(temperature_shortfalls, index=dates, columns=REFERENCE_TEMPERATURES).groupby(level=0).mean()
    weather_hour_counts = temperature.groupby(dates).size()
    daily_degree_days = daily_degree_days.loc[weather_hour_counts[weather_hour_counts >= MINIMUM_WEATHER_HOURS].index]
    daily_degree_days.index = pd.to_datetime(daily_degree_days.index)
    return daily_degree_days

In [ ]:
@dataclass(frozen=True)
class GasProfile:
    balance_temperature_c: float
    slope_kwh_per_degree_day: float
    base_kwh_per_day: float
    r_squared: float
    days: int
    tied_candidates: int
    tied_low_c: float
    tied_high_c: float

    def estimate_gas(self, heating_degree_days):
        return np.maximum(
            self.base_kwh_per_day
            + self.slope_kwh_per_degree_day * heating_degree_days[self.balance_temperature_c].to_numpy(),
            0.0,
        )

In [ ]:
def fit_gas_profile(gas, heating_degree_days):
    gas_use = np.ascontiguousarray(gas, dtype=float)
    weather_demand = np.ascontiguousarray(heating_degree_days, dtype=float)
    mean_weather_demand, mean_gas_use = weather_demand.mean(axis=0), gas_use.mean()
    centred_demand = weather_demand - mean_weather_demand
    variance = np.square(centred_demand).sum(axis=0)
    covariance = (centred_demand * (gas_use - mean_gas_use)[:, None]).sum(axis=0)
    slopes = np.where(variance > 0, covariance / np.where(variance > 0, variance, 1.0), 0.0)
    background_estimates = mean_gas_use - slopes * mean_weather_demand
    demand_squares = np.square(weather_demand).sum(axis=0)
    zero_background_slopes = np.maximum(
        (weather_demand * gas_use[:, None]).sum(axis=0) / np.where(demand_squares > 0, demand_squares, 1.0), 0.0
    )
    negative_base = background_estimates < 0
    slopes[negative_base], background_estimates[negative_base] = zero_background_slopes[negative_base], 0.0
    negative_slope = slopes < 0
    slopes[negative_slope], background_estimates[negative_slope] = 0.0, max(mean_gas_use, 0.0)
    squared_errors = np.square(gas_use[:, None] - (background_estimates[None, :] + slopes[None, :] * weather_demand)).sum(axis=0)
    tied = np.flatnonzero(np.isclose(squared_errors, squared_errors.min(), rtol=1e-10, atol=1e-8))
    selected_candidate = int(tied[0])
    total_gas_variation = np.square(gas_use - mean_gas_use).sum()
    return GasProfile(
        float(heating_degree_days.columns[selected_candidate]), float(slopes[selected_candidate]), float(background_estimates[selected_candidate]),
        float(1.0 - squared_errors[selected_candidate] / total_gas_variation) if total_gas_variation > 0 else np.nan, len(gas_use),
        len(tied), float(heating_degree_days.columns[tied[0]]), float(heating_degree_days.columns[tied[-1]])
    )

In [ ]:
def estimate_gas_by_method(training_gas, training_degree_days, assessment_degree_days, gas_profile=None):
    gas_profile = gas_profile or fit_gas_profile(training_gas, training_degree_days)
    fixed_degree_days = training_degree_days[15.5].to_numpy()
    gas_use = training_gas.to_numpy()
    variance = np.square(fixed_degree_days - fixed_degree_days.mean()).sum()
    slope = (
        ((fixed_degree_days - fixed_degree_days.mean()) * (gas_use - gas_use.mean())).sum() / variance
        if variance > 0 else 0.0
    )
    background_gas = gas_use.mean() - slope * fixed_degree_days.mean()
    return {
        COMPARISON_METHODS[0]: gas_profile.estimate_gas(assessment_degree_days),
        COMPARISON_METHODS[1]: np.full(len(assessment_degree_days), gas_use.mean()),
        COMPARISON_METHODS[2]: np.maximum(background_gas + slope * assessment_degree_days[15.5].to_numpy(), 0.0),
    }

In [ ]:
def compare_daily_estimates(household_inputs, minimum_training_days=MINIMUM_TRAINING_DAYS,
                           minimum_assessment_days=MINIMUM_ASSESSMENT_DAYS):
    monthly_results, gas_estimates = [], []
    for home, household_data in household_inputs.items():
        gas, weather = household_data["gas"], household_data["degree_days"]
        dates = gas.index
        for month in sorted(dates.to_period("M").unique()):
            training_days, assessment_days = dates < month.start_time, dates.to_period("M") == month
            if training_days.sum() < minimum_training_days or assessment_days.sum() < minimum_assessment_days:
                continue
            gas_profile = fit_gas_profile(gas.loc[training_days], weather.loc[training_days])
            estimates = estimate_gas_by_method(gas.loc[training_days], weather.loc[training_days], weather.loc[assessment_days], gas_profile)
            observed_gas = gas.loc[assessment_days].to_numpy()
            for method, estimated_gas in estimates.items():
                monthly_results.append({
                    "home": home, "month": str(month), "model": method,
                    "training_days": int(training_days.sum()), "test_days": int(assessment_days.sum()),
                    "mae_kwh_day": float(np.abs(estimated_gas - observed_gas).mean()),
                    "rmse_kwh_day": float(np.sqrt(np.square(estimated_gas - observed_gas).mean())),
                    "balance_temperature_c": gas_profile.balance_temperature_c if method == COMPARISON_METHODS[0] else np.nan,
                    "tied_candidates": gas_profile.tied_candidates if method == COMPARISON_METHODS[0] else np.nan,
                    "tied_low_c": gas_profile.tied_low_c if method == COMPARISON_METHODS[0] else np.nan,
                    "tied_high_c": gas_profile.tied_high_c if method == COMPARISON_METHODS[0] else np.nan,
                })
                gas_estimates.extend({
                    "home": home, "date": str(date.date()), "month": str(month), "model": method,
                    "actual_kwh": float(observed_value), "predicted_kwh": float(estimated_value),
                } for date, observed_value, estimated_value in zip(dates[assessment_days], observed_gas, estimated_gas))
    return pd.DataFrame(monthly_results), pd.DataFrame(gas_estimates)

In [ ]:
def load_daily_inputs(folder):
    folder = Path(folder)
    for item in json.loads((folder / "input_manifest.json").read_text()):
        if hashlib.sha256((folder / item["file"]).read_bytes()).hexdigest() != item["sha256"]:
            raise ValueError(f"Input changed: {item['file']}")
    excluded_dates = {
        "H1": {"2025-12-01", "2026-03-29", "2026-04-21", "2026-04-22"}
        | {f"2026-02-{day:02d}" for day in range(5, 11)},
        "H2": {"2026-04-30", "2026-08-11"},
    }
    household_inputs, coverage = {}, []
    for home in ("H1", "H2"):
        daily_gas = pd.read_csv(
            folder / f"{home.lower()}_daily_gas.csv", parse_dates=["date"], float_precision="round_trip"
        ).set_index("date")
        eligible_dates = daily_gas.observations.ge(20 if home == "H1" else 46)
        eligible_dates &= ~daily_gas.index.strftime("%Y-%m-%d").isin(excluded_dates[home])
        hourly_weather = pd.read_csv(folder / f"{home.lower()}_hourly_weather.csv", float_precision="round_trip")
        hourly_weather.index = pd.to_datetime(hourly_weather.timestamp_utc, utc=True).dt.tz_convert("Europe/London")
        temperature = hourly_weather.temperature_c
        daily_degree_days = calculate_degree_days(temperature)
        dates = daily_gas.index[eligible_dates].intersection(daily_degree_days.index)
        gas = daily_gas.loc[dates, "gas_kwh"]
        if gas.isna().any() or gas.lt(0).any() or not dates.is_unique:
            raise ValueError(f"Invalid daily observations for {home}")
        household_inputs[home] = {
            "gas": gas, "degree_days": daily_degree_days.loc[dates], "temperature": temperature,
            "weather": "Local outdoor sensor" if home == "H1" else "ERA5 gridded weather",
        }
        coverage.append({
            "home": home, "source_gas_days": len(daily_gas), "gas_days_after_rules": int(eligible_dates.sum()),
            "matched_days": len(dates), "first_day": str(dates.min().date()),
            "last_day": str(dates.max().date()), "weather": household_inputs[home]["weather"],
        })
    return household_inputs, pd.DataFrame(coverage)

In [ ]:
if reproduce_study:
    household_inputs, daily_coverage = load_daily_inputs(INPUTS_FOLDER / "daily")
    daily_coverage.to_csv(DAILY_OUTPUTS / "coverage.csv", index=False)
    display(HTML(daily_coverage.to_html(index=False)))

In [ ]:
def plot_pipe_and_gas(pipes, gas):
    set_observation_chart_style()
    figure, axes = plt.subplots(2, 1, figsize=(7.2, 3.75), sharex=True, gridspec_kw={"height_ratios": [2.2, 1]})
    figure.subplots_adjust(left=.13, right=.955, top=.79, bottom=.13, hspace=.12)
    figure.text(.02, .95, "Recorded pipe temperatures and gas use", fontsize=13, weight="bold")
    for channel, colour, style in [("flow", OBSERVATION_DARK, "-"), ("return", OBSERVATION_MID, "--")]:
        observations = pipes[channel]
        continuous_periods = observations["timestamp"].diff().gt(pd.Timedelta(minutes=15)).cumsum()
        for period_number, (_, period_observations) in enumerate(observations.groupby(continuous_periods)):
            axes[0].plot(period_observations["timestamp"], period_observations["temperature_c"], color=colour, linestyle=style,
                         linewidth=1.4, marker=".", markersize=1.6, label=f"{channel.title()} pipe" if period_number == 0 else None)
    axes[0].set_ylabel("Pipe temperature (°C)")
    axes[0].set_ylim(10, 92)
    axes[0].set_yticks([20, 40, 60, 80])
    axes[0].legend(loc="upper left", bbox_to_anchor=(0, 1.18), frameon=False, ncol=2, fontsize=8.5)
    interval_widths = (gas["interval_end"] - gas["interval_start"]).dt.total_seconds() / 86400
    axes[1].bar(gas["interval_start"], gas["kwh"], width=interval_widths, align="edge", color="#777777", edgecolor="white", linewidth=.7)
    axes[1].set_ylabel("Gas input\n(kWh / 30 min)")
    axes[1].set_ylim(0, 9)
    axes[1].set_yticks([0, 4, 8])
    axes[1].set_xlim(gas["interval_start"].min(), gas["interval_end"].max())
    axes[1].xaxis.set_major_locator(mdates.HourLocator())
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz=gas["interval_start"].dt.tz))
    axes[1].set_xlabel("Time (UTC)")
    for axis in axes:
        axis.grid(axis="y", color="#DEDEDE", linewidth=.7)
        axis.set_axisbelow(True)
    return save_observation_figure(figure, "F02_pipe_and_gas")

In [ ]:
if reproduce_study:
    pipe_figure = plot_pipe_and_gas(pipe_observations, gas_intervals)
    plt.close(pipe_figure)
    display(Image(filename=str(FIGURES_FOLDER / "F02_pipe_and_gas.png")))

In [ ]:
if reproduce_study:
    cooling_start = pd.Timestamp("2026-01-02 12:00", tz="UTC")
    cooling_end = pd.Timestamp("2026-01-02 13:00", tz="UTC")
    cooling_rows = []
    for channel, observations in pipe_observations.items():
        hour_observations = observations.loc[observations.timestamp.ge(cooling_start) & observations.timestamp.lt(cooling_end)]
        cooling_rows.append({"pipe": channel, "first_record_c": hour_observations.temperature_c.iloc[0],
                             "last_record_c": hour_observations.temperature_c.iloc[-1], "records": len(hour_observations)})
    cooling_summary = pd.DataFrame(cooling_rows)
    cooling_summary.to_csv(OBSERVATION_OUTPUTS / "cooling_interval.csv", index=False)
    display(cooling_summary.rename(columns={"pipe": "Pipe", "first_record_c": "First observation (°C)",
                                    "last_record_c": "Last observation (°C)", "records": "Observations"})
            .style.hide(axis="index").format(precision=1).set_caption("Pipe temperature summary"))
    display(gas_intervals.loc[gas_intervals.interval_start.ge(cooling_start) & gas_intervals.interval_end.le(cooling_end)]
            .style.hide(axis="index").format(precision=1).set_caption("Gas consumption during the same hour"))

In [ ]:
if reproduce_study:
    household_profiles = {home: fit_gas_profile(household_data["gas"], household_data["degree_days"]) for home, household_data in household_inputs.items()}
    profile_summary = pd.DataFrame([{"home": home, **asdict(gas_profile)} for home, gas_profile in household_profiles.items()])
    profile_summary.to_csv(DAILY_OUTPUTS / "fits.csv", index=False)
    display(HTML(profile_summary[["home", "days", "balance_temperature_c", "base_kwh_per_day", "slope_kwh_per_degree_day"]].round(3).to_html(index=False)))

In [ ]:
def set_model_chart_style():
    plt.rcParams.update({
        "font.family": "DejaVu Sans", "font.size": 8, "axes.titlesize": 9,
        "axes.labelsize": 8, "xtick.labelsize": 7.5, "ytick.labelsize": 7.5,
        "text.color": "#303030", "axes.labelcolor": "#303030",
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.edgecolor": "#A0A0A0", "grid.color": "#DEDEDE", "grid.linewidth": 0.6,
        "legend.frameon": False, "savefig.facecolor": "white",
    })

In [ ]:
def save_model_figure(figure, figure_name, figure_folder):
    for extension in ("png", "svg"):
        figure.savefig(figure_folder / f"{figure_name}.{extension}", dpi=300, bbox_inches="tight")

In [ ]:
def plot_profiles(household_inputs, profiles, output_folder, figure_folder):
    observations = []
    figure, axes = plt.subplots(1, 2, figsize=(6.30, 2.9), sharex=True, sharey=True)
    for axis, (home, household_data) in zip(axes, household_inputs.items()):
        gas_profile = profiles[home]
        heating_demand = household_data["degree_days"][gas_profile.balance_temperature_c]
        axis.scatter(heating_demand, household_data["gas"], s=9, color=MODEL_MID, alpha=0.8, edgecolor="none")
        demand_range = np.array([0.0, heating_demand.max()])
        axis.plot(demand_range, gas_profile.base_kwh_per_day + gas_profile.slope_kwh_per_degree_day * demand_range,
                  color=MODEL_DARK, linewidth=1.6)
        axis.set_title(f"{home} · {gas_profile.days} matched days", loc="left", weight="bold")
        axis.text(0.04, 0.96, f"Balance: {gas_profile.balance_temperature_c:g}°C\n"
                  f"Base: {gas_profile.base_kwh_per_day:.1f} kWh/day\n"
                  f"Slope: {gas_profile.slope_kwh_per_degree_day:.2f}",
                  transform=axis.transAxes, ha="left", va="top", fontsize=7)
        axis.set_xlabel("Cold-weather demand (degree days)")
        axis.set_xlim(left=0)
        axis.set_xticks([0, 5, 10, 15])
        axis.set_ylim(bottom=0)
        axis.grid(axis="y")
        observations.extend({
            "home": home, "date": str(date.date()), "gas_kwh": float(gas),
            "degree_days": float(heating_demand.loc[date]), "balance_temperature_c": gas_profile.balance_temperature_c,
            "predicted_kwh": float(predicted), "weather": household_data["weather"],
        } for date, gas, predicted in zip(household_data["gas"].index, household_data["gas"],
                                         gas_profile.estimate_gas(household_data["degree_days"])))
    axes[0].set_ylabel("Daily gas (kWh)")
    figure.tight_layout(w_pad=1.2)
    pd.DataFrame(observations).to_csv(output_folder / "F03_profiles.csv", index=False)
    save_model_figure(figure, "F03", figure_folder)
    return figure

In [ ]:
if reproduce_study:
    set_model_chart_style()
    profile_figure = plot_profiles(household_inputs, household_profiles, DAILY_OUTPUTS, FIGURES_FOLDER)
    plt.close(profile_figure)
    display(Image(filename=str(FIGURES_FOLDER / "F03.png")))

In [ ]:
if reproduce_study:
    monthly_gas_results, daily_gas_estimates = compare_daily_estimates(household_inputs)
    daily_summary = monthly_gas_results.groupby(["home", "model"], as_index=False).agg(
        months=("month", "size"), mean_monthly_mae_kwh_day=("mae_kwh_day", "mean")
    )
    monthly_gas_results.to_csv(DAILY_OUTPUTS / "monthly_folds.csv", index=False)
    daily_gas_estimates.to_csv(DAILY_OUTPUTS / "later_month_predictions.csv", index=False)
    daily_summary.to_csv(DAILY_OUTPUTS / "score_summary.csv", index=False)

In [ ]:
def plot_estimates(gas_estimates, output_folder, figure_folder):
    selected_estimates = gas_estimates.loc[gas_estimates.model.eq(COMPARISON_METHODS[0])].copy()
    selected_estimates.to_csv(output_folder / "F04_later_days.csv", index=False)
    figure, axes = plt.subplots(2, 1, figsize=(6.30, 4.1), sharex=True, sharey=True)
    for axis, (home, rows) in zip(axes, selected_estimates.groupby("home")):
        rows["date"] = pd.to_datetime(rows.date)
        calendar_estimates = rows.set_index("date").reindex(pd.date_range(rows.date.min(), rows.date.max()))
        axis.plot(calendar_estimates.index, calendar_estimates.actual_kwh, color=MODEL_DARK, linewidth=0.95, label="Observed gas")
        axis.plot(calendar_estimates.index, calendar_estimates.predicted_kwh, color=MODEL_MID, linestyle="--",
                  linewidth=1.05, label="Estimate using observed weather")
        axis.set_title(f"{home} · {len(rows)} scored days", loc="left", weight="bold")
        axis.set_ylabel("Daily gas (kWh)")
        axis.set_ylim(bottom=0)
        axis.grid(axis="y")
    axes[0].legend(loc="upper right", fontsize=7, ncol=1)
    axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    axes[-1].set_xlabel("2026")
    figure.tight_layout(h_pad=1.2)
    save_model_figure(figure, "F04", figure_folder)
    return figure

In [ ]:
if reproduce_study:
    set_model_chart_style()
    estimate_figure = plot_estimates(daily_gas_estimates, DAILY_OUTPUTS, FIGURES_FOLDER)
    plt.close(estimate_figure)
    display(Image(filename=str(FIGURES_FOLDER / "F04.png")))

In [ ]:
def plot_monthly_errors(monthly_results, output_folder, figure_folder):
    monthly_results.to_csv(output_folder / "F05_monthly_errors.csv", index=False)
    figure, axes = plt.subplots(1, 2, figsize=(6.30, 3.5), sharex=True, sharey=True)
    method_names = (COMPARISON_METHODS[0], COMPARISON_METHODS[2], COMPARISON_METHODS[1])
    labels = ("Fitted balance", "Fixed 15.5°C", "Earlier mean")
    for axis, (home, rows) in zip(axes, monthly_results.groupby("home")):
        months = sorted(rows.month.unique())
        for offset, name, label, colour, marker in zip(
            (-0.18, 0, 0.18), method_names, labels, (MODEL_DARK, MODEL_MID, MODEL_GREY), ("o", "s", "x")
        ):
            method_errors = rows.loc[rows.model.eq(name)].set_index("month").loc[months]
            axis.scatter(method_errors.mae_kwh_day, np.arange(len(months)) + offset,
                         color=colour, marker=marker, s=23, label=label, zorder=3)
        axis.set_yticks(np.arange(len(months)), [pd.Period(month).strftime("%b") for month in months])
        axis.set_title(home, loc="left", weight="bold")
        axis.set_xlim(left=0)
        axis.grid(axis="x")
        axis.set_xlabel("Mean absolute error (kWh/day)")
    axes[0].invert_yaxis()
    axes[0].set_ylabel("2026 test month")
    handles, labels = axes[0].get_legend_handles_labels()
    figure.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.54, 1.02), ncol=3, fontsize=7)
    figure.tight_layout(rect=(0, 0, 1, 0.94), w_pad=1.2)
    save_model_figure(figure, "F05", figure_folder)
    return figure

In [ ]:
if reproduce_study:
    set_model_chart_style()
    error_figure = plot_monthly_errors(monthly_gas_results, DAILY_OUTPUTS, FIGURES_FOLDER)
    plt.close(error_figure)
    display(Image(filename=str(FIGURES_FOLDER / "F05.png")))

In [ ]:
if reproduce_study:
    error_table = daily_summary.pivot(index="model", columns="home", values="mean_monthly_mae_kwh_day")
    error_table = error_table.loc[[COMPARISON_METHODS[0], COMPARISON_METHODS[2], COMPARISON_METHODS[1]]]
    error_table.index = error_table.index.str.replace("balance temperature", "reference").str.replace("15.5 C", "15.5 °C")
    error_table = error_table.rename(columns={"H1": "H1 MAE (kWh/day)", "H2": "H2 MAE (kWh/day)"})
    display(error_table.round(2))

In [ ]:
def load_hourly_weather(folder):
    folder = Path(folder)
    manifest = json.loads((folder / "input_manifest.json").read_text())
    for item in manifest["inputs"]:
        if hashlib.sha256((folder / item["input_file"]).read_bytes()).hexdigest() != item["sha256"]:
            raise ValueError(f"Input changed: {item['input_file']}")
    hourly_inputs = pd.read_parquet(folder / "h1_hourly_inputs.parquet")
    if not hourly_inputs.index.is_unique or str(hourly_inputs.index.tz) != "UTC":
        raise ValueError("Weather comparison requires unique UTC hourly timestamps")
    return hourly_inputs

In [ ]:
def prepare_weather_inputs(hourly_inputs):
    gas_increments = hourly_inputs.gas_increment_kwh.dropna()
    daily_gas = gas_increments.resample("1D").sum(min_count=1)
    daily_gas = daily_gas.loc[gas_increments.resample("1D").count() >= 20]
    daily_gas.index = daily_gas.index.tz_localize(None)
    weather_degree_days = {name: calculate_degree_days(hourly_inputs[column]) for name, column in WEATHER_SOURCES.items()}
    matched_dates = daily_gas.index
    for source_degree_days in weather_degree_days.values():
        matched_dates = matched_dates.intersection(source_degree_days.index)
    matched_dates = matched_dates.sort_values()
    return daily_gas.loc[matched_dates], {name: source_degree_days.loc[matched_dates] for name, source_degree_days in weather_degree_days.items()}

In [ ]:
def monthly_evaluation_dates(dates):
    for month in sorted(dates.to_period("M").unique()):
        training_dates = dates[dates < month.start_time]
        assessment_dates = dates[dates.to_period("M") == month]
        if len(training_dates) >= 60 and len(assessment_dates) >= 10:
            yield str(month), training_dates, assessment_dates

In [ ]:
def compare_weather_sources(gas, weather):
    monthly_results, gas_estimates = [], []
    for month, training_dates, assessment_dates in monthly_evaluation_dates(gas.index):
        for source, source_degree_days in weather.items():
            gas_profile = fit_gas_profile(gas.loc[training_dates], source_degree_days.loc[training_dates])
            estimated_gas = gas_profile.estimate_gas(source_degree_days.loc[assessment_dates])
            observed_gas = gas.loc[assessment_dates].to_numpy()
            monthly_results.append({
                "month": month, "source": source, "training_days": len(training_dates),
                "test_days": len(assessment_dates), "training_start": str(training_dates.min().date()),
                "training_end": str(training_dates.max().date()), "test_start": str(assessment_dates.min().date()),
                "test_end": str(assessment_dates.max().date()),
                "mae_kwh_day": float(np.abs(observed_gas - estimated_gas).mean()), **asdict(gas_profile),
            })
            gas_estimates.extend({
                "date": str(date.date()), "month": month, "source": source,
                "actual_kwh": float(observed), "predicted_kwh": float(predicted),
                "absolute_error_kwh": float(abs(observed - predicted)),
            } for date, observed, predicted in zip(assessment_dates, observed_gas, estimated_gas))
    return pd.DataFrame(monthly_results), pd.DataFrame(gas_estimates)

In [ ]:
def summarise_weather_results(gas_estimates):
    return gas_estimates.groupby("source", sort=False).agg(
        mae_kwh_day=("absolute_error_kwh", "mean"), test_days=("date", "nunique"),
        months=("month", "nunique"),
    ).reset_index()

In [ ]:
if reproduce_study:
    hourly_weather_inputs = load_hourly_weather(INPUTS_FOLDER / "comparison")
    weather_comparison_gas, weather_degree_days = prepare_weather_inputs(hourly_weather_inputs)
    monthly_weather_results, weather_estimates = compare_weather_sources(weather_comparison_gas, weather_degree_days)
    weather_summary = summarise_weather_results(weather_estimates)
    monthly_weather_results.to_csv(WEATHER_OUTPUTS / "monthly_fits.csv", index=False)
    weather_estimates.to_csv(WEATHER_OUTPUTS / "daily_estimates.csv", index=False)
    weather_summary.to_csv(WEATHER_OUTPUTS / "summary.csv", index=False)
    display(HTML(weather_summary.round(4).to_html(index=False)))

In [ ]:
def plot_weather_comparison(monthly_results, summary, folder):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    monthly_errors = monthly_results.pivot(index="month", columns="source", values="mae_kwh_day")
    monthly_day_counts = monthly_results.groupby("month").test_days.first()
    chart_data = monthly_errors.copy()
    chart_data.loc["All scored days"] = summary.set_index("source").mae_kwh_day
    chart_data["test_days"] = list(monthly_day_counts) + [int(summary.test_days.iloc[0])]
    chart_data.index.name = "period"
    chart_data.to_csv(folder / "F06_weather_comparison.csv", index=True)
    figure, axis = plt.subplots(figsize=(9.2, 4.3))
    row_positions = np.arange(len(chart_data))
    for name, colour, marker, offset in (
        ("Local outdoor", "#404040", "o", -0.12), ("ERA5", "#707070", "s", 0.12)
    ):
        axis.scatter(chart_data[name], row_positions + offset, s=45, color=colour, marker=marker, label=name, zorder=3)
        for index, mean_error in enumerate(chart_data[name]):
            axis.annotate(f"{mean_error:.2f}", (mean_error, index + offset), xytext=(7, 0),
                          textcoords="offset points", va="center", fontsize=9, color=colour)
    labels = [f"{pd.Timestamp(month).strftime('%b')}  ({int(monthly_day_counts.loc[month])} days)" for month in monthly_errors.index]
    axis.set_yticks(row_positions, labels + [f"All scored days  ({int(summary.test_days.iloc[0])})"])
    axis.invert_yaxis()
    axis.set_xlim(0, float(monthly_errors.max().max()) + 3.0)
    axis.set_xlabel("Mean absolute error (kWh/day)")
    axis.axhline(len(chart_data) - 1.5, color="#C5C5C5", linewidth=0.8)
    axis.grid(axis="x", color="#DEDEDE", linewidth=0.8)
    axis.set_axisbelow(True)
    axis.spines[["top", "right", "left"]].set_visible(False)
    axis.spines["bottom"].set_color("#A0A0A0")
    axis.tick_params(axis="y", length=0)
    axis.legend(loc="upper right", frameon=False)
    figure.suptitle("Daily gas estimates by weather source", x=0.02, ha="left",
                   fontsize=16, fontweight="bold", color="#303030")
    figure.tight_layout(rect=(0, 0, 1, 0.94))
    for extension in ("png", "svg"):
        figure.savefig(folder / f"F06_weather_comparison.{extension}", dpi=220, bbox_inches="tight")
    plt.close(figure)
    return chart_data.reset_index()

In [ ]:
if reproduce_study:
    set_model_chart_style()
    weather_chart = plot_weather_comparison(monthly_weather_results, weather_summary, FIGURES_FOLDER)
    display(Image(filename=str(FIGURES_FOLDER / "F06_weather_comparison.png")))

In [ ]:
def read_pipe_temperatures():
    channels = {}
    for name in ("flow", "return"):
        pipe_observations = pd.read_parquet(SENSOR_INPUTS / f"h2_{name}.parquet")
        temperatures = pipe_observations.set_index(pd.to_datetime(pipe_observations.timestamp, utc=True)).value.sort_index()
        assert temperatures.index.is_unique and np.isfinite(temperatures).all()
        channels[name] = temperatures
    return channels

In [ ]:
def summarise_pipe_half_hours(channels):
    shared_start = max(temperatures.index.min() for temperatures in channels.values())
    shared_end = min(temperatures.index.max() for temperatures in channels.values())
    minute_temperatures = pd.concat({name: temperatures.resample("1min").mean() for name, temperatures in channels.items()}, axis=1)
    minute_temperatures = minute_temperatures.loc[shared_start.floor("min"):shared_end.floor("min")]
    minute_temperatures["gap"] = minute_temperatures.flow - minute_temperatures["return"]
    half_hour_summary = minute_temperatures.resample("30min").agg(
        paired_minutes=("gap", "count"),
        flow_mean_C=("flow", "mean"),
        return_mean_C=("return", "mean"),
        gap_mean_C=("gap", "mean"),
    )
    half_hour_summary["fully_inside_shared_span"] = (half_hour_summary.index >= shared_start) & (half_hour_summary.index + pd.Timedelta(minutes=30) <= shared_end)
    return half_hour_summary

In [ ]:
def prepare_gas_intervals(gas_intervals):
    gas_intervals = gas_intervals.copy()
    for column in ("interval_start", "interval_end"):
        gas_intervals[column] = pd.to_datetime(gas_intervals[column]).dt.tz_localize(
            "Europe/London", ambiguous="NaT", nonexistent="NaT"
        ).dt.tz_convert("UTC")
    valid_intervals = gas_intervals.interval_start.notna() & gas_intervals.interval_end.notna()
    valid_intervals &= (gas_intervals.interval_end - gas_intervals.interval_start).eq(pd.Timedelta(minutes=30))
    valid_intervals &= gas_intervals.interval_start.eq(gas_intervals.interval_start.dt.floor("30min"))
    valid_intervals &= np.isfinite(gas_intervals.kwh) & gas_intervals.kwh.ge(0)
    gas = gas_intervals.loc[valid_intervals].set_index("interval_start").kwh.rename("gas_kwh")
    assert gas.index.is_unique
    return gas, int((~valid_intervals).sum())

In [ ]:
def select_complete_intervals(coverage, gas):
    return coverage.loc[coverage.paired_minutes.eq(30)].join(gas, how="inner").dropna(subset=["gas_kwh"])

In [ ]:
def analyse_pipe_sensor_value():
    coverage = summarise_pipe_half_hours(read_pipe_temperatures())
    gas, excluded_interval_count = prepare_gas_intervals(pd.read_parquet(SENSOR_INPUTS / "h2_gas_intervals.parquet"))
    sample = select_complete_intervals(coverage, gas)
    correlations = pd.DataFrame([
        {"feature": field, "description": label, "pearson_r": sample[field].corr(sample.gas_kwh), "half_hours": len(sample)}
        for field, label in SENSOR_MEASURES.items()
    ])
    later_intervals = coverage.loc[(coverage.index >= LATER_PERIOD_START) & coverage.fully_inside_shared_span].join(gas)
    summary = {
        "half_hours": len(sample), "UTC_dates": sample.index.normalize().nunique(),
        "first_start_UTC": str(sample.index.min()), "last_start_UTC": str(sample.index.max()),
        "positive_gas_bins": int(sample.gas_kwh.gt(0).sum()), "zero_gas_bins": int(sample.gas_kwh.eq(0).sum()),
        "shared_clock_bins_including_partial_edges": len(coverage), "selected_share": len(sample) / len(coverage),
        "excluded_invalid_gas_intervals": excluded_interval_count, "correlations": correlations.to_dict("records"),
        "later_start_UTC": str(LATER_PERIOD_START), "later_full_clock_bins": len(later_intervals),
        "later_gas_bins": int(later_intervals.gas_kwh.notna().sum()),
        "later_zero_gas_bins": int(later_intervals.gas_kwh.eq(0).sum()),
        "later_positive_gas_bins": int(later_intervals.gas_kwh.gt(0).sum()),
        "later_complete_temperature_bins": int(later_intervals.paired_minutes.eq(30).sum()),
        "later_max_paired_minutes": int(later_intervals.paired_minutes.max()),
        "scope": "Selected same-half-hour association; no prediction model. Later observations were already inspected during development, not an untouched test.",
    }
    SENSOR_OUTPUTS.mkdir(parents=True, exist_ok=True)
    sample.to_csv(SENSOR_OUTPUTS / "F07_matched_half_hours.csv", index_label="UTC_interval_start")
    coverage.join(gas).to_csv(SENSOR_OUTPUTS / "half_hour_coverage.csv", index_label="UTC_interval_start")
    later_intervals.to_csv(SENSOR_OUTPUTS / "later_coverage.csv", index_label="UTC_interval_start")
    correlations.to_csv(SENSOR_OUTPUTS / "correlations.csv", index=False)
    (SENSOR_OUTPUTS / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    return sample, summary

In [ ]:
if reproduce_study:
    sensor_sample, sensor_summary = analyse_pipe_sensor_value()
    sensor_selection = pd.DataFrame([{
        "Selected half-hours": sensor_summary["half_hours"], "UTC dates": sensor_summary["UTC_dates"],
        "Positive gas": sensor_summary["positive_gas_bins"], "Zero gas": sensor_summary["zero_gas_bins"],
        "Full shared-span bins": sensor_summary["shared_clock_bins_including_partial_edges"],
        "Selected share (%)": 100 * sensor_summary["selected_share"],
    }])
    display(HTML(sensor_selection.round(3).to_html(index=False)))

In [ ]:
if reproduce_study:
    display(HTML(pd.DataFrame(sensor_summary["correlations"]).round(6).to_html(index=False)))

In [ ]:
def plot_sensor_comparison(sample):
    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11, "text.color": "#303030",
                         "axes.labelcolor": "#303030", "xtick.color": "#303030", "ytick.color": "#303030",
                         "svg.hashsalt": "sensor-value-exploration"})
    figure, axes = plt.subplots(1, 2, figsize=(11.5, 5.25), sharey=True)
    figure.subplots_adjust(left=.09, right=.98, bottom=.23, top=.80, wspace=.15)
    figure.text(.09, .94, "One pipe sensor and two", fontsize=19, weight="bold")
    months = sample.index.strftime("%Y-%m")
    groups = [("2026-01", "January (35)", "#404040", "o"), ("2026-02", "February (32)", "#777777", "^"),
              ("other", "Other months (9)", "#555555", "s")]
    for axis, (field, label), title in zip(axes, SENSOR_MEASURES.items(), ("A  One sensor", "B  Two sensors")):
        for month, group, colour, marker in groups:
            selected_months = ~pd.Index(months).isin(["2026-01", "2026-02"]) if month == "other" else months == month
            monthly_observations = sample.loc[selected_months]
            axis.scatter(monthly_observations[field], monthly_observations.gas_kwh, s=39, color=colour, marker=marker,
                       alpha=.95, edgecolors="#303030", linewidth=.35, label=group)
        axis.set_title(title, loc="left", fontsize=13, weight="bold", pad=12)
        axis.set_xlabel(label + " (°C)", labelpad=9)
        axis.set_ylim(0, 10.25)
        axis.set_yticks(range(0, 11, 2))
        axis.spines[["top", "right"]].set_visible(False)
        axis.spines[["left", "bottom"]].set_color("#A0A0A0")
        axis.grid(axis="y", color="#DEDEDE", linewidth=.7)
        axis.set_axisbelow(True)
        axis.text(.04, .94, f"Pearson r = {sample[field].corr(sample.gas_kwh):.3f}", transform=axis.transAxes,
                va="top", bbox={"facecolor": "white", "edgecolor": "none", "alpha": .9})
    axes[0].set_ylabel("Gas in the same half-hour (kWh)", labelpad=9)
    handles, labels = axes[0].get_legend_handles_labels()
    figure.legend(handles, labels, loc="center left", bbox_to_anchor=(.09, .055), ncol=3, frameon=False)
    for extension in ("png", "svg"):
        export_settings = {"metadata": {"Date": None}} if extension == "svg" else {"dpi": 240}
        figure.savefig(ANALYSIS_FOLDER / f"figures/F07_sensor_value.{extension}", facecolor="white", **export_settings)
    return figure

In [ ]:
if reproduce_study:
    sensor_figure = plot_sensor_comparison(sensor_sample)
    plt.close(sensor_figure)
    display(Image(filename=str(FIGURES_FOLDER / "F07_sensor_value.png")))

In [ ]:
if reproduce_study:
    later_coverage = pd.read_csv(SENSOR_OUTPUTS / "later_coverage.csv", index_col=0, parse_dates=True)
    later_counts = pd.Series({
        "Full later half-hours": len(later_coverage),
        "Valid gas observation": later_coverage.gas_kwh.notna().sum(),
        "Zero gas": later_coverage.gas_kwh.eq(0).sum(),
        "Positive gas": later_coverage.gas_kwh.gt(0).sum(),
        "No valid gas observation": later_coverage.gas_kwh.isna().sum(),
        "Thirty paired minutes": later_coverage.paired_minutes.eq(30).sum(),
        "Best paired-minute count": later_coverage.paired_minutes.max(),
    }, name="Count").astype(int)
    later_counts.to_csv(FIGURES_FOLDER / "F10_sample_coverage.csv", index_label="Measure")

In [ ]:
if reproduce_study:
    def plot_later_coverage(later_intervals):
        paired_minute_counts = later_intervals.paired_minutes
        groups = {
            "A  Gas observations": pd.Series({
                "Zero gas": int(later_intervals.gas_kwh.eq(0).sum()),
                "Positive gas": int(later_intervals.gas_kwh.gt(0).sum()),
                "No valid gas observation": int(later_intervals.gas_kwh.isna().sum()),
            }),
            "B  Paired temperature observations": pd.Series({
                "No paired minutes": int(paired_minute_counts.eq(0).sum()),
                "1-29 paired minutes": int(paired_minute_counts.between(1, 29).sum()),
                "All 30 paired minutes": int(paired_minute_counts.eq(30).sum()),
            }),
        }
        assert paired_minute_counts.between(0, 30).all()
        assert all(counts.sum() == len(later_intervals) for counts in groups.values())
        plot_data = pd.concat(groups, names=["Panel", "Category"]).rename("Half-hours")
        plot_data.to_csv(FIGURES_FOLDER / "F10_plot_data.csv")
        figure, axes = plt.subplots(1, 2, figsize=(10.8, 3.5), sharex=True)
        for axis, (title, counts) in zip(axes, groups.items()):
            bars = axis.barh(range(len(counts)), counts.values, height=.52, color="#404040")
            axis.set_yticks(range(len(counts)), counts.index, fontsize=12)
            axis.invert_yaxis()
            axis.set_xlim(0, len(later_intervals) * 1.08)
            axis.set_xticks([0, 1000, 2000, 3000], ["0", "1,000", "2,000", "3,000"])
            axis.set_xlabel("Number of half-hours", fontsize=12)
            axis.set_title(title, loc="left", fontsize=13, weight="bold", pad=14)
            axis.bar_label(bars, labels=[f"{value:,}" for value in counts], padding=5, fontsize=12)
            axis.grid(axis="y", visible=False)
            axis.set_axisbelow(True)
            axis.spines[["top", "right", "left"]].set_visible(False)
            axis.tick_params(axis="y", length=0)
        figure.tight_layout(w_pad=2.5)
        for extension in ("png", "svg"):
            figure.savefig(FIGURES_FOLDER / f"F10_sample_coverage.{extension}", dpi=220, bbox_inches="tight")
        return figure

    set_model_chart_style()
    later_figure = plot_later_coverage(later_coverage)
    plt.close(later_figure)
    display(Image(filename=str(FIGURES_FOLDER / "F10_sample_coverage.png")))

In [ ]:
if reproduce_study:
    from pathlib import Path
    import hashlib
    import json
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    ROOM_SENSOR_CONFIGURATION = {'start': '2025-12-08T00:00:00+00:00',
     'end_exclusive': '2026-08-12T00:00:00+00:00',
     'fit_cutoff': '2026-03-01T00:00:00+00:00',
     'minimum_forecast_hours': 168,
     'minimum_forecast_days': 14,
     'boundary_times': {'H1': ['2025-12-10T16:07:56+00:00',
                               '2026-01-09T16:07:56+00:00',
                               '2026-02-08T16:07:56+00:00',
                               '2026-03-10T16:07:56+00:00',
                               '2026-04-09T15:07:56+00:00',
                               '2026-05-09T15:07:56+00:00',
                               '2026-06-08T15:07:56+00:00',
                               '2026-07-08T15:07:56+00:00',
                               '2026-08-07T15:07:56+00:00'],
                        'H2': ['2026-01-09T16:06:35+00:00',
                               '2026-02-08T16:06:35+00:00',
                               '2026-03-10T16:06:35+00:00',
                               '2026-04-09T15:06:35+00:00',
                               '2026-05-09T15:06:35+00:00',
                               '2026-06-08T15:06:35+00:00',
                               '2026-07-08T15:06:35+00:00',
                               '2026-08-07T15:06:35+00:00']},
     'forecast_candidates': [{'target': 'H1_living_room',
                              'extra': 'H1_outdoor',
                              'family': 'primary_room_outdoor'},
                             {'target': 'H1_dining_room',
                              'extra': 'H1_outdoor',
                              'family': 'primary_room_outdoor'},
                             {'target': 'H1_kitchen',
                              'extra': 'H1_outdoor',
                              'family': 'primary_room_outdoor'},
                             {'target': 'H1_office',
                              'extra': 'H1_outdoor',
                              'family': 'primary_room_outdoor'},
                             {'target': 'H1_master_bedroom',
                              'extra': 'H1_outdoor',
                              'family': 'primary_room_outdoor'},
                             {'target': 'H1_bedroom_1',
                              'extra': 'H1_outdoor',
                              'family': 'primary_room_outdoor'},
                             {'target': 'H2_living_room',
                              'extra': 'H2_outdoor',
                              'family': 'primary_room_outdoor'},
                             {'target': 'H2_playroom',
                              'extra': 'H2_outdoor',
                              'family': 'primary_room_outdoor'},
                             {'target': 'H2_kitchen',
                              'extra': 'H2_outdoor',
                              'family': 'primary_room_outdoor'},
                             {'target': 'H2_bathroom',
                              'extra': 'H2_outdoor',
                              'family': 'primary_room_outdoor'},
                             {'target': 'H2_bedroom_1',
                              'extra': 'H2_outdoor',
                              'family': 'primary_room_outdoor'},
                             {'target': 'H2_master_bedroom',
                              'extra': 'H2_outdoor',
                              'family': 'primary_room_outdoor'}],
     'roster': ['H1_living_room',
                'H1_dining_room',
                'H1_kitchen',
                'H1_office',
                'H1_master_bedroom',
                'H1_bedroom_1',
                'H2_living_room',
                'H2_playroom',
                'H2_kitchen',
                'H2_bathroom',
                'H2_bedroom_1',
                'H2_master_bedroom',
                'H1_outdoor',
                'H2_outdoor'],
     'invalid_intervals': {}}


In [ ]:
def room_sensor_read_inputs(source_folder):
    source_folder = Path(source_folder)
    manifest = json.loads((source_folder / "inputs/input_manifest.json").read_text())
    required = set(ROOM_SENSOR_CONFIGURATION["roster"])
    records = {}
    for item in manifest["files"]:
        if item["alias"] not in required:
            continue
        path = source_folder / item["path"]
        assert hashlib.sha256(path.read_bytes()).hexdigest() == item["sha256"]
        frame = pd.read_parquet(path)
        frame["timestamp"] = pd.to_datetime(frame["timestamp"], utc=True)
        assert frame["timestamp"].notna().all()
        assert not frame["timestamp"].duplicated().any()
        records[item["alias"]] = frame.sort_values("timestamp").reset_index(drop=True)
    assert set(records) == required
    return records


In [ ]:
def room_sensor_align_sensor(frame, grid, maximum_age, boundaries, invalid_interval=None):
    available = frame.loc[~(frame.timestamp.isin(boundaries) & frame.state_numeric.notna())].copy()
    if invalid_interval:
        bad = available.timestamp.between(*invalid_interval, inclusive='left')
        available.loc[bad, 'state_numeric'] = np.nan
    right = available.rename(columns={'timestamp': 'source_time', 'state_numeric': 'value'})
    result = pd.merge_asof(pd.DataFrame({'time': grid}), right[['source_time', 'value']], left_on='time', right_on='source_time', direction='backward', tolerance=pd.Timedelta(minutes=maximum_age)).set_index('time')
    if invalid_interval:
        result.loc[result.index.to_series().between(*invalid_interval, inclusive='left'), 'value'] = np.nan
    result['age_minutes'] = (result.index.to_series() - result.source_time).dt.total_seconds() / 60
    return result


In [ ]:
def room_sensor_align_all(records, protocol, maximum_age):
    grid = pd.date_range(protocol['start'], protocol['end_exclusive'], freq='h', inclusive='left')
    aligned = {}
    for (name, frame) in records.items():
        boundaries = pd.to_datetime(protocol['boundary_times'][name[:2]], utc=True)
        invalid = protocol['invalid_intervals'].get(name)
        interval = tuple(pd.to_datetime(invalid, utc=True)) if invalid else None
        aligned[name] = room_sensor_align_sensor(frame, grid, maximum_age, boundaries, interval)
    return aligned


In [ ]:
def room_sensor_forecast_frame(aligned, target, extra, require_future):
    (room, other) = (aligned[target], aligned[extra])
    frame = pd.DataFrame({'current': room.value, 'past_change': room.value.diff(), 'extra': other.value, 'seasonal': room.value.shift(23), 'target': room.value.shift(-1), 'source_current': room.source_time, 'source_extra': other.source_time, 'source_past': room.source_time.shift(1), 'source_seasonal': room.source_time.shift(23), 'source_target': room.source_time.shift(-1)})
    frame['hour_sin'] = np.sin(2 * np.pi * frame.index.hour / 24)
    frame['hour_cos'] = np.cos(2 * np.pi * frame.index.hour / 24)
    frame['target_time'] = frame.index + pd.Timedelta(hours=1)
    frame['future_report'] = frame.source_target > frame.index.to_series()
    frame['actual_elapsed_minutes'] = (frame.source_target - frame.source_current).dt.total_seconds() / 60
    frame['target_lead_minutes'] = (frame.source_target - frame.index.to_series()).dt.total_seconds() / 60
    complete = frame.dropna()
    selected = complete.loc[complete.future_report] if require_future else complete
    audit = {'complete_hours': len(complete), 'selected_hours': len(selected), 'already_available_label_fraction': (~complete.future_report).mean() if len(complete) else np.nan}
    return (selected, audit)


In [ ]:
def room_sensor_design_matrix(frame, extra=False):
    columns = ['current', 'past_change', 'hour_sin', 'hour_cos'] + (['extra'] if extra else [])
    return np.column_stack([np.ones(len(frame)), frame[columns].to_numpy(float)])


In [ ]:
def room_sensor_fit_linear(train, extra=False):
    matrix = room_sensor_design_matrix(train, extra)
    (coefficients, residuals, rank, singular_values) = np.linalg.lstsq(matrix, train.target.to_numpy(), rcond=None)
    return (coefficients, int(rank))


In [ ]:
def room_sensor_block_range(errors, minimum_blocks=8, draws=1000):
    values = errors.copy()
    values['week'] = values.index.tz_localize(None).to_period('W-SUN').start_time
    blocks = values.groupby('week')['gain'].agg(['sum', 'count'])
    if len(blocks) < minimum_blocks:
        return (np.nan, np.nan, len(blocks))
    rng = np.random.default_rng(20260906)
    indices = rng.integers(0, len(blocks), size=(draws, len(blocks)))
    scores = blocks['sum'].to_numpy()[indices].sum(axis=1) / blocks['count'].to_numpy()[indices].sum(axis=1)
    (low, high) = np.quantile(scores, [0.025, 0.975])
    return (low, high, len(blocks))


In [ ]:
def room_sensor_evaluate_forecasts(aligned, protocol, variant):
    (result, monthly, predictions, audits) = ([], [], [], [])
    cutoff = pd.Timestamp(protocol['fit_cutoff'])
    for candidate in protocol['forecast_candidates']:
        (target, extra) = (candidate['target'], candidate['extra'])
        (frame, audit) = room_sensor_forecast_frame(aligned, target, extra, variant != '60_held')
        (before_filter, before_audit) = room_sensor_forecast_frame(aligned, target, extra, False)
        later_before = before_filter.loc[before_filter.index >= cutoff]
        train = frame.loc[(frame.target_time < cutoff) & (frame.source_target < cutoff)]
        later = frame.loc[frame.index >= cutoff]
        row = {'variant': variant, 'target_sensor': target, 'extra_sensor': extra, 'family': candidate['family'], 'train_hours': len(train), 'test_hours': len(later), 'train_dates': train.index.normalize().nunique(), 'test_dates': later.index.normalize().nunique()}
        audits.append({**row, **audit, 'later_complete_before_future_filter': len(later_before), 'later_known_target_count_before_filter': int((~later_before.future_report).sum()), 'later_known_target_fraction_before_filter': (~later_before.future_report).mean() if len(later_before) else np.nan, 'later_already_available_label_fraction': (~later.future_report).mean() if len(later) else np.nan, 'target_lead_minutes_median': later.target_lead_minutes.median(), 'target_lead_minutes_p05': later.target_lead_minutes.quantile(0.05), 'target_lead_minutes_p95': later.target_lead_minutes.quantile(0.95), 'elapsed_minutes_median': later.actual_elapsed_minutes.median(), 'elapsed_minutes_p05': later.actual_elapsed_minutes.quantile(0.05), 'elapsed_minutes_p95': later.actual_elapsed_minutes.quantile(0.95)})
        enough = min(len(train), len(later)) >= protocol['minimum_forecast_hours']
        enough = enough and min(row['train_dates'], row['test_dates']) >= protocol['minimum_forecast_days']
        if not enough:
            result.append({**row, 'status': 'insufficient earlier or later overlap'})
            continue
        (base, base_rank) = room_sensor_fit_linear(train)
        (extended, extra_rank) = room_sensor_fit_linear(train, True)
        prediction = pd.DataFrame({'actual': later.target, 'persistence': later.current, 'seasonal': later.seasonal, 'indoor_only': room_sensor_design_matrix(later) @ base, 'with_extra': room_sensor_design_matrix(later, True) @ extended}, index=later.index)
        error = prediction.drop(columns='actual').sub(prediction.actual, axis=0).abs()
        error['gain'] = error.indoor_only - error.with_extra
        error['gain_over_persistence'] = error.persistence - error.with_extra
        (low, high, weeks) = room_sensor_block_range(error)
        result.append({**row, 'status': 'available', 'base_rank': base_rank, 'extra_rank': extra_rank, **{name + '_mae_c': error[name].mean() for name in ['persistence', 'seasonal', 'indoor_only', 'with_extra']}, 'gain_c': error.gain.mean(), 'gain_over_persistence_c': error.gain_over_persistence.mean(), 'gain_block_low_c': low, 'gain_block_high_c': high, 'weekly_blocks': weeks, 'range_status': 'descriptive weekly-block range' if weeks >= 8 else 'unavailable: fewer than eight weekly blocks'})
        for (month, part) in error.groupby(error.index.strftime('%Y-%m')):
            monthly.append({**row, 'month': month, 'hours': len(part), **{name + '_mae_c': part[name].mean() for name in ['persistence', 'seasonal', 'indoor_only', 'with_extra']}, 'gain_c': part.gain.mean(), 'gain_over_persistence_c': part.gain_over_persistence.mean()})
        prediction['target_sensor'] = target
        prediction['extra_sensor'] = extra
        prediction['variant'] = variant
        prediction['time'] = prediction.index
        predictions.append(prediction.reset_index(drop=True))
    return (pd.DataFrame(result), pd.DataFrame(monthly), pd.concat(predictions, ignore_index=True), pd.DataFrame(audits))


In [ ]:
def room_sensor_plot_forecast_gains(scores, candidates, figure_folder):
    selected = scores.loc[(scores.variant == '60_future') & (scores.family == 'primary_room_outdoor')]
    (fig, axes) = plt.subplots(1, 2, figsize=(12, 6), sharex=True)
    for (home, ax) in zip(['H1', 'H2'], axes):
        part = selected.loc[selected.target_sensor.str.startswith(home)].reset_index(drop=True)
        for (i, row) in part.iterrows():
            if row.status != 'available':
                continue
            ax.plot([row.gain_block_low_c, row.gain_block_high_c], [i, i], c='#777777', lw=3)
            ax.scatter(row.gain_c, i, c='#404040', s=45)
            ax.scatter(row.gain_over_persistence_c, i, c='#777777', marker='s', s=35)
        ax.set_yticks(range(len(part)), part.target_sensor.str[3:].str.replace('_', ' '))
        ax.invert_yaxis()
        ax.axvline(0, c='#707070', lw=1)
        ax.set_title(home)
        ax.set_xlabel('Reduction in later mean absolute error (°C)')
        ax.grid(axis='x', alpha=0.15)
    axes[0].scatter([], [], c='#404040', label='Compared with indoor-only model')
    axes[0].scatter([], [], c='#777777', marker='s', label='Compared with holding current temperature')
    fig.legend(loc='lower center', bbox_to_anchor=(0.5, -0.09), frameon=False, ncol=2)
    fig.suptitle('Outdoor temperature and next-hour forecast error', fontsize=15, fontweight='bold')
    fig.tight_layout()
    room_sensor_save_figure(fig, figure_folder, '04_outdoor_predictive_value')


In [ ]:
def room_sensor_validate_timing(records, aligned, protocol):
    validation_results = []

    def record_validation(name, passed):
        assert bool(passed), name
        validation_results.append({'check': name, 'passed': bool(passed)})
    for (name, data) in aligned.items():
        valid = data.value.notna()
        record_validation(name + ': unique regular hourly grid', data.index.is_unique and (data.index.to_series().diff().dropna() == pd.Timedelta(hours=1)).all())
        record_validation(name + ': no future source', (data.loc[valid].source_time <= data.index[valid]).all())
        record_validation(name + ': bounded record age', data.loc[valid].age_minutes.between(0, 60).all())
    cutoff = pd.Timestamp(protocol['fit_cutoff'])
    for candidate in protocol['forecast_candidates']:
        (target, extra) = (candidate['target'], candidate['extra'])
        (frame, audit) = room_sensor_forecast_frame(aligned, target, extra, True)
        train = frame.loc[(frame.target_time < cutoff) & (frame.source_target < cutoff)]
        later = frame.loc[frame.index >= cutoff]
        label = target + '+' + extra
        record_validation(label + ': source target after origin', (frame.source_target > frame.index.to_series()).all())
        record_validation(label + ': training labels before cutoff', (train.target_time < cutoff).all() and (train.source_target < cutoff).all())
        record_validation(label + ': known previous and seasonal readings', (frame.source_past <= frame.index - pd.Timedelta(hours=1)).all() and (frame.source_seasonal <= frame.index - pd.Timedelta(hours=23)).all())
        if len(train) < 168 or len(later) < 168:
            continue
        (beta, rank) = room_sensor_fit_linear(train, True)
        initial_predictions = room_sensor_design_matrix(later, True) @ beta
        changed_labels = later.copy()
        changed_labels['target'] += 20
        record_validation(label + ': held-out label mutation cannot change predictions', np.array_equal(room_sensor_design_matrix(changed_labels, True) @ beta, initial_predictions))
        record_validation(label + ': held-out label mutation changes score', not np.isclose(np.mean(np.abs(changed_labels.target - initial_predictions)), np.mean(np.abs(later.target - initial_predictions))))
        changed_train = train.copy()
        changed_train['target'] += 5
        (new_beta, new_rank) = room_sensor_fit_linear(changed_train, True)
        record_validation(label + ': training-label positive control', np.allclose(room_sensor_design_matrix(later, True) @ new_beta, initial_predictions + 5, atol=1e-07))
        for origin in later.index[[0, len(later) // 2, -1]]:
            local_grid = pd.date_range(origin - pd.Timedelta(hours=23), origin, freq='h')
            changed = {}
            for sensor in [target, extra]:
                raw = records[sensor].copy()
                raw.loc[raw.timestamp > origin, 'state_numeric'] = 9999
                appended = pd.DataFrame({'timestamp': [raw.timestamp.max() + pd.Timedelta(days=2)], 'state_numeric': [8888.0]})
                raw = pd.concat([raw, appended], ignore_index=True).sort_values('timestamp')
                boundaries = pd.to_datetime(protocol['boundary_times'][sensor[:2]], utc=True)
                interval = protocol['invalid_intervals'].get(sensor)
                invalid = tuple(pd.to_datetime(interval, utc=True)) if interval else None
                changed[sensor] = room_sensor_align_sensor(raw, local_grid, 60, boundaries, invalid)
                record_validation(label + str(origin) + ': future mutation and append preserve past ' + sensor, changed[sensor].value.equals(aligned[sensor].loc[local_grid].value))
            row = later.loc[[origin]].copy()
            row['current'] = changed[target].loc[origin, 'value']
            row['past_change'] = changed[target].value.iloc[-1] - changed[target].value.iloc[-2]
            row['extra'] = changed[extra].loc[origin, 'value']
            row['seasonal'] = changed[target].value.iloc[0]
            record_validation(label + str(origin) + ': future mutation preserves origin prediction', np.allclose(room_sensor_design_matrix(row, True) @ beta, room_sensor_design_matrix(later.loc[[origin]], True) @ beta))
    small_grid = pd.date_range('2026-01-01', periods=5, freq='h', tz='UTC')
    small = pd.DataFrame({'timestamp': small_grid[[0, 1, 3]], 'state_numeric': [20.0, np.nan, 22.0]})
    sampled = room_sensor_align_sensor(small, small_grid, 120, pd.DatetimeIndex([], tz='UTC'))
    record_validation('unavailable state stops carry until a new numeric record', sampled.value.iloc[1:3].isna().all())
    record_validation('no backward filling from a later numeric record', pd.isna(sampled.value.iloc[2]))
    known = sampled.copy()
    small.loc[0, 'state_numeric'] += 1
    moved = room_sensor_align_sensor(small, small_grid, 120, pd.DatetimeIndex([], tz='UTC'))
    record_validation('past input positive control reaches a nonzero-coefficient prediction', moved.value.iloc[0] * 2 == known.value.iloc[0] * 2 + 2)
    return pd.DataFrame(validation_results)


In [ ]:
def room_sensor_validate_score_integrity(aligned30, protocol, scores, predictions):
    validation_results = []

    def record_validation(name, passed):
        assert bool(passed), name
        validation_results.append({'check': name, 'passed': bool(passed)})
    for (name, data) in aligned30.items():
        valid = data.value.notna()
        record_validation(name + ': strict 30-minute record age', data.loc[valid].age_minutes.between(0, 30).all())
    for candidate in protocol['forecast_candidates']:
        (frame, audit) = room_sensor_forecast_frame(aligned30, candidate['target'], candidate['extra'], True)
        record_validation(candidate['target'] + '+' + candidate['extra'] + ': strict target source is future', (frame.source_target > frame.index.to_series()).all())
    for row in scores.loc[scores.status == 'available'].itertuples():
        part = predictions.loc[(predictions.variant == row.variant) & (predictions.target_sensor == row.target_sensor) & (predictions.extra_sensor == row.extra_sensor)]
        prefix = row.variant + ':' + row.target_sensor + '+' + row.extra_sensor
        record_validation(prefix + ': identical target rows across all four predictions', part[['actual', 'persistence', 'seasonal', 'indoor_only', 'with_extra']].notna().all().all() and len(part) == row.test_hours and part.time.is_unique)
        for method in ['persistence', 'seasonal', 'indoor_only', 'with_extra']:
            expected = (part[method] - part.actual).abs().mean()
            record_validation(prefix + ': ' + method + ' score reconciles', np.isclose(expected, getattr(row, method + '_mae_c'), atol=1e-12))
    return pd.DataFrame(validation_results)


In [ ]:
def room_sensor_save_figure(figure, figure_folder, name):
    figure_folder = Path(figure_folder)
    figure_folder.mkdir(parents=True, exist_ok=True)
    figure.savefig(figure_folder / (name + ".png"), bbox_inches="tight", facecolor="white")
    figure.savefig(figure_folder / (name + ".svg"), bbox_inches="tight", facecolor="white")
    plt.close(figure)


In [ ]:
def run_room_sensor_comparison(source_folder, output_folder, figure_folder):
    source_folder = Path(source_folder)
    output_folder = Path(output_folder)
    figure_folder = Path(figure_folder)
    output_folder.mkdir(parents=True, exist_ok=True)
    figure_folder.mkdir(parents=True, exist_ok=True)
    records = room_sensor_read_inputs(source_folder)
    configuration = ROOM_SENSOR_CONFIGURATION
    alignment = {age: room_sensor_align_all(records, configuration, age) for age in (60, 30)}
    results = {
        variant: room_sensor_evaluate_forecasts(alignment[age], configuration, variant)
        for variant, age in [("60_future", 60), ("60_held", 60), ("30_future", 30)]
    }
    scores = pd.concat([result[0] for result in results.values()], ignore_index=True)
    months = pd.concat([result[1] for result in results.values()], ignore_index=True)
    predictions = pd.concat([result[2] for result in results.values()], ignore_index=True)
    sample_audit = pd.concat([result[3] for result in results.values()], ignore_index=True)
    assert scores.groupby("variant").size().eq(12).all()
    scores.to_csv(output_folder / "forecast_scores.csv", index=False)
    months.to_csv(output_folder / "forecast_months.csv", index=False)
    predictions.to_parquet(output_folder / "forecast_predictions.parquet", index=False)
    sample_audit.to_csv(output_folder / "forecast_sample_audit.csv", index=False)
    timing_validation = room_sensor_validate_timing(records, alignment[60], configuration)
    score_validation = room_sensor_validate_score_integrity(alignment[30], configuration, scores, predictions)
    timing_validation.to_csv(output_folder / "timing_and_mutation.csv", index=False)
    score_validation.to_csv(output_folder / "score_and_sensitivity.csv", index=False)
    with plt.rc_context({"figure.dpi":120, "savefig.dpi":180, "font.size":10,
                         "axes.spines.top":False, "axes.spines.right":False,
                         "axes.titleweight":"bold", "axes.labelcolor":"#303030",
                         "text.color":"#303030", "axes.edgecolor":"#9A9A9A",
                         "xtick.color":"#303030", "ytick.color":"#303030",
                         "grid.color":"#DEDEDE"}):
        room_sensor_plot_forecast_gains(scores, configuration["forecast_candidates"], figure_folder)
    primary = scores.loc[scores.variant.eq("60_future")]
    counts = pd.Series({
        "Outdoor improves indoor-only model": int(primary.gain_c.gt(0).sum()),
        "Holding current temperature beats outdoor model": int(primary.gain_over_persistence_c.lt(0).sum())
    }, name="Rooms (out of 12)")
    return {"scores": scores, "monthly_errors": months, "predictions": predictions,
            "sample_audit": sample_audit, "timing_validation": timing_validation,
            "score_validation": score_validation, "comparison_counts": counts,
            "figure_path": figure_folder / "04_outdoor_predictive_value.png"}


In [ ]:
if reproduce_study:
    room_analysis = run_room_sensor_comparison(
        ANALYSIS_FOLDER / "supplementary/sensor_relationships",
        ANALYSIS_FOLDER / "outputs/room_sensor_comparison",
        FIGURES_FOLDER,
    )
    room_scores = room_analysis["scores"].query("variant == '60_future'")
    room_comparison_counts = room_analysis["comparison_counts"]
    assert len(room_scores) == room_scores.target_sensor.nunique() == 12
    assert room_scores.status.eq("available").all()
    assert room_comparison_counts.tolist() == [10, 11]

In [ ]:
if reproduce_study:
    display(Image(filename=str(room_analysis["figure_path"])))

In [ ]:
if reproduce_study:
    CALIBRATION_WINDOW_DAYS = (14, 30, 60, 90, 120, 180)
    ASSESSMENT_DAYS = 28

    def summarise_temperature_coverage(temperature, dates, reference_temperature):
        retained_temperatures = temperature.loc[pd.to_datetime(temperature.index.date).isin(dates)].dropna()
        daily_temperatures = retained_temperatures.groupby(pd.to_datetime(retained_temperatures.index.date)).agg(["min", "max", "mean", "count"])
        return {
            "minimum_hourly_c": float(retained_temperatures.min()),
            "maximum_hourly_c": float(retained_temperatures.max()),
            "hourly_p10_c": float(retained_temperatures.quantile(0.1)),
            "hourly_p90_c": float(retained_temperatures.quantile(0.9)),
            "all_hours_below_days": int(daily_temperatures["max"].lt(reference_temperature).sum()),
            "crossing_days": int((daily_temperatures["min"].lt(reference_temperature) & daily_temperatures["max"].ge(reference_temperature)).sum()),
            "all_hours_at_or_above_days": int(daily_temperatures["min"].ge(reference_temperature).sum()),
            "retained_hours": len(retained_temperatures),
        }

In [ ]:
def assess_calibration_window(household_data, training_start, assessment_start, assessment_end):
    gas, weather = household_data["gas"], household_data["degree_days"]
    training_dates = gas.index[(gas.index >= training_start) & (gas.index < assessment_start)]
    assessment_dates = gas.index[(gas.index >= assessment_start) & (gas.index < assessment_end)]
    training_gas, training_degree_days = gas.loc[training_dates], weather.loc[training_dates]
    assessment_degree_days = weather.loc[assessment_dates]
    gas_profile = fit_gas_profile(training_gas, training_degree_days)
    estimates = estimate_gas_by_method(training_gas, training_degree_days, assessment_degree_days, gas_profile)
    coverage = summarise_temperature_coverage(household_data["temperature"], training_dates, gas_profile.balance_temperature_c)
    fixed_coverage = summarise_temperature_coverage(household_data["temperature"], training_dates, 15.5)
    candidate_profiles = [fit_gas_profile(training_gas, training_degree_days[[reference]]) for reference in REFERENCE_TEMPERATURES]
    squared_errors = np.array([np.square(training_gas.to_numpy() - candidate_profile.estimate_gas(training_degree_days)).sum()
                       for candidate_profile in candidate_profiles])
    tied_profiles = [candidate_profile for candidate_profile, squared_error in zip(candidate_profiles, squared_errors)
            if np.isclose(squared_error, squared_errors.min(), rtol=1e-10, atol=1e-8)]
    tied_estimates = np.stack([candidate_profile.estimate_gas(assessment_degree_days) for candidate_profile in tied_profiles])
    profile_details = {
        **asdict(gas_profile),
        "training_days": len(training_dates),
        "test_days": len(assessment_dates),
        "first_training_day": str(training_dates.min().date()),
        "last_training_day": str(training_dates.max().date()),
        "first_test_day": str(assessment_dates.min().date()),
        "last_test_day": str(assessment_dates.max().date()),
        "base_tied_low": min(candidate_profile.base_kwh_per_day for candidate_profile in tied_profiles),
        "base_tied_high": max(candidate_profile.base_kwh_per_day for candidate_profile in tied_profiles),
        "slope_tied_low": min(candidate_profile.slope_kwh_per_degree_day for candidate_profile in tied_profiles),
        "slope_tied_high": max(candidate_profile.slope_kwh_per_degree_day for candidate_profile in tied_profiles),
        "mean_tied_test_spread_kwh": float(np.ptp(tied_estimates, axis=0).mean()),
        "max_tied_test_spread_kwh": float(np.ptp(tied_estimates, axis=0).max()),
        "candidate_references_above_all_training_hours": int((REFERENCE_TEMPERATURES > coverage["maximum_hourly_c"]).sum()),
        **{f"fitted_reference_{key}": value for key, value in coverage.items()},
        **{f"fixed_15_5_{key}": value for key, value in fixed_coverage.items()},
    }
    observed = gas.loc[assessment_dates].to_numpy()
    scores = [{"model": model, "mae_kwh_day": float(np.abs(estimated - observed).mean()),
               "rmse_kwh_day": float(np.sqrt(np.square(estimated - observed).mean()))}
              for model, estimated in estimates.items()]
    gas_estimates = [{"model": model, "date": str(date.date()), "actual_kwh": float(actual),
                    "estimated_kwh": float(estimated)}
                   for model, estimated in estimates.items()
                   for date, actual, estimated in zip(assessment_dates, observed, estimated)]
    return profile_details, scores, gas_estimates

In [ ]:
def calibration_windows(household_data):
    first_date, last_date = household_data["gas"].index.min(), household_data["gas"].index.max()
    for assessment_start in pd.date_range("2026-01-01", "2026-07-01", freq="MS"):
        for days in CALIBRATION_WINDOW_DAYS:
            yield "matched_cutoff", assessment_start - pd.Timedelta(days=days), assessment_start, days
    maximum_history_days = int((last_date + pd.Timedelta(days=1) - first_date).days) - ASSESSMENT_DAYS
    for days in sorted(set(range(14, maximum_history_days + 1, 7)) | {window_days for window_days in CALIBRATION_WINDOW_DAYS if window_days <= maximum_history_days}):
        yield "winter_start", first_date, first_date + pd.Timedelta(days=days), days

In [ ]:
def analyse_calibration(household_inputs):
    profile_results, scores, gas_estimates, eligibility_rows = [], [], [], []
    for home, household_data in household_inputs.items():
        dates = household_data["gas"].index
        for design, training_start, assessment_start, calendar_days in calibration_windows(household_data):
            assessment_end = assessment_start + pd.Timedelta(days=ASSESSMENT_DAYS)
            training_days = int(((dates >= training_start) & (dates < assessment_start)).sum())
            test_days = int(((dates >= assessment_start) & (dates < assessment_end)).sum())
            eligibility_status = ("precedes_record" if training_start < dates.min() else
                      "incomplete_future_window" if assessment_end > dates.max() + pd.Timedelta(days=1) else
                      "too_few_training_days" if training_days < 10 else
                      "too_few_test_days" if test_days < 20 else "eligible")
            window_details = {"home": home, "design": design, "start": str(training_start.date()),
                       "cutoff": str(assessment_start.date()), "test_end_exclusive": str(assessment_end.date()),
                       "calendar_days": calendar_days}
            eligibility_rows.append({**window_details, "training_days": training_days, "test_days": test_days, "status": eligibility_status})
            if eligibility_status != "eligible":
                continue
            profile_details, window_scores, window_estimates = assess_calibration_window(household_data, training_start, assessment_start, assessment_end)
            profile_results.append({**window_details, **profile_details})
            scores.extend({**window_details, **row, "training_days": training_days, "test_days": test_days}
                          for row in window_scores)
            gas_estimates.extend({**window_details, **row} for row in window_estimates)
    return {"fits": pd.DataFrame(profile_results), "scores": pd.DataFrame(scores),
            "predictions": pd.DataFrame(gas_estimates), "eligibility": pd.DataFrame(eligibility_rows)}

In [ ]:
if reproduce_study:
    CALIBRATION_OUTPUTS = ANALYSIS_FOLDER / "outputs/calibration"
    CALIBRATION_VALIDATION = ANALYSIS_FOLDER / "checks/calibration"
    for folder in [CALIBRATION_OUTPUTS, CALIBRATION_VALIDATION]:
        folder.mkdir(parents=True, exist_ok=True)
    calibration_results = analyse_calibration(household_inputs)
    for name, table in calibration_results.items():
        table.to_csv(CALIBRATION_OUTPUTS / f"{name}.csv", index=False)

In [ ]:
if reproduce_study:
    def validate_calibration(household_inputs, results):
        passed_rules = []

        def validate_rule(rule_name, condition):
            if not condition:
                raise AssertionError(rule_name)
            passed_rules.append(rule_name)

        profile_results, scores, gas_estimates = (results[rule_name] for rule_name in ("fits", "scores", "predictions"))
        window_keys = ["home", "design", "start", "cutoff", "calendar_days"]
        validate_rule("Unique replay keys", not profile_results.duplicated(window_keys).any())
        validate_rule("All fitted observations precede the test cutoff",
              bool((pd.to_datetime(profile_results.last_training_day) < pd.to_datetime(profile_results.cutoff)).all()))
        validate_rule("All test dates lie inside the following 28 calendar days",
              bool(((pd.to_datetime(gas_estimates.date) >= pd.to_datetime(gas_estimates.cutoff)) &
                    (pd.to_datetime(gas_estimates.date) < pd.to_datetime(gas_estimates.test_end_exclusive))).all()))
        validate_rule("Coverage categories partition eligible training days",
              all(bool((profile_results[[f"{prefix}_all_hours_below_days", f"{prefix}_crossing_days",
                               f"{prefix}_all_hours_at_or_above_days"]].sum(axis=1) == profile_results.training_days).all())
                  for prefix in ["fitted_reference", "fixed_15_5"]))
        matched_estimates = gas_estimates.loc[gas_estimates.design.eq("matched_cutoff")]
        validate_rule("Each matched cutoff uses identical test dates for every duration and method",
              all(len({tuple(sorted(method_rows.date)) for _, method_rows in group.groupby(["calendar_days", "model"])}) == 1
                  for _, group in matched_estimates.groupby(["home", "cutoff"])))
        calculated_errors = gas_estimates.assign(error=lambda estimate_rows: abs(estimate_rows.actual_kwh - estimate_rows.estimated_kwh))
        calculated_errors = calculated_errors.groupby(window_keys + ["model"]).error.mean().sort_index()
        reported_errors = scores.set_index(window_keys + ["model"]).mae_kwh_day.sort_index()
        validate_rule("MAE reconciles to all saved prediction rows", np.allclose(calculated_errors, reported_errors, rtol=1e-12, atol=1e-12))
        household_data = household_inputs["H2"]
        training_start, assessment_start, assessment_end = pd.to_datetime(["2026-05-02", "2026-07-01", "2026-07-29"])
        original_details, _, original_estimates = assess_calibration_window(household_data, training_start, assessment_start, assessment_end)
        changed_household = copy.deepcopy(household_data)
        changed_household["gas"].loc[changed_household["gas"].index >= assessment_start] += 1000
        altered_details, _, altered_estimates = assess_calibration_window(changed_household, training_start, assessment_start, assessment_end)
        validate_rule("Later gas cannot change earlier fitted parameters", original_details == altered_details)
        validate_rule("Later gas cannot change predictions",
              [estimate_row["estimated_kwh"] for estimate_row in original_estimates] == [estimate_row["estimated_kwh"] for estimate_row in altered_estimates])
        changed_household = copy.deepcopy(household_data)
        changed_household["degree_days"].loc[changed_household["degree_days"].index >= assessment_start] += 5
        altered_details, _, altered_estimates = assess_calibration_window(changed_household, training_start, assessment_start, assessment_end)
        parameter_keys = ["base_kwh_per_day", "slope_kwh_per_degree_day", "balance_temperature_c",
                          "tied_low_c", "tied_high_c", "training_days"]
        validate_rule("Later weather cannot change earlier fitted parameters",
              all(original_details[key] == altered_details[key] for key in parameter_keys))
        validate_rule("Changing legitimate test weather can change estimates",
              [estimate_row["estimated_kwh"] for estimate_row in original_estimates] != [estimate_row["estimated_kwh"] for estimate_row in altered_estimates])
        changed_household = copy.deepcopy(household_data)
        changed_household["gas"].loc[(changed_household["gas"].index >= training_start) & (changed_household["gas"].index < assessment_start)] += 20
        altered_details, _, _ = assess_calibration_window(changed_household, training_start, assessment_start, assessment_end)
        validate_rule("Changing training gas can change the fit", original_details["base_kwh_per_day"] != altered_details["base_kwh_per_day"])
        example_results = []
        example_scenarios = [("cold_only", np.arange(10), 50 - 3 * np.arange(10)),
                 ("three_days", np.array([5, 14, 20]), np.array([35, 8, 5])),
                 ("one_day_plus_3", np.array([5, 14, 20]), np.array([35, 11, 5]))]
        for label, temperatures, gas in example_scenarios:
            weather = pd.DataFrame(np.maximum(REFERENCE_TEMPERATURES[None, :] - temperatures[:, None], 0), columns=REFERENCE_TEMPERATURES)
            gas_profile = fit_gas_profile(pd.Series(gas), weather)
            example_results.append({"case": label, "reference_c": gas_profile.balance_temperature_c,
                        "background_kwh_day": gas_profile.base_kwh_per_day,
                        "slope_kwh_degree_day": gas_profile.slope_kwh_per_degree_day,
                        "tied_low_c": gas_profile.tied_low_c, "tied_high_c": gas_profile.tied_high_c,
                        "training_rmse": float(np.sqrt(np.square(gas - gas_profile.estimate_gas(weather)).mean()))})
        validate_rule("Cold-only toy has multiple equally fitting references", example_results[0]["tied_high_c"] > example_results[0]["tied_low_c"])
        validate_rule("Three noiseless mixed days recover the constructed reference", example_results[1]["reference_c"] == 15)
        validate_rule("One perturbed toy day changes the reference despite another exact fit",
              example_results[2]["reference_c"] == 16.25 and example_results[1]["training_rmse"] < 1e-10 and example_results[2]["training_rmse"] < 1e-10)
        return {"status": "passed", "checks_passed": len(passed_rules), "checks": passed_rules,
                "scope": "Replay arithmetic, matched dates, information boundaries and constructed identification examples; no physical calibration or independent field validation.",
                "synthetic_examples": example_results}

    calibration_validation = validate_calibration(household_inputs, calibration_results)
    (CALIBRATION_VALIDATION / "results.json").write_text(json.dumps(calibration_validation, indent=2) + "\n");

In [ ]:
if reproduce_study:
    def plot_calibration_profiles(profile_history, figure_path):
        palette = {"H1": "#404040", "H2": "#777777"}
        metrics = [("base_kwh_per_day", "base_tied_low", "base_tied_high", "Background gas\n(kWh/day)"),
                   ("slope_kwh_per_degree_day", "slope_tied_low", "slope_tied_high", "Weather response\n(kWh per degree day)"),
                   ("balance_temperature_c", "tied_low_c", "tied_high_c", "Reference temperature\n(°C)")]
        with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 10, "text.color": "#303030",
                             "axes.labelcolor": "#303030", "xtick.color": "#303030", "ytick.color": "#303030"}):
            figure, axes = plt.subplots(3, 2, figsize=(10, 7.2), sharex=True, sharey="row")
            for column, home in enumerate(["H1", "H2"]):
                household_history = profile_history.loc[profile_history.design.eq("winter_start") & profile_history.home.eq(home)].sort_values("calendar_days")
                for row, (measure, lower_limit, upper_limit, label) in enumerate(metrics):
                    axis = axes[row, column]
                    axis.fill_between(household_history.calendar_days, household_history[lower_limit], household_history[upper_limit], color=palette[home], alpha=0.18, linewidth=0)
                    axis.plot(household_history.calendar_days, household_history[measure], color=palette[home], marker="o" if home == "H1" else "s", markersize=3, linewidth=1.6)
                    axis.spines[["top", "right"]].set_visible(False)
                    axis.grid(axis="y", color="#DEDEDE", linewidth=0.7)
                    axis.set_axisbelow(True)
                    axis.set_xlim(10, 222)
                    axis.set_xticks([14, 60, 120, 180, 210])
                    if column == 0:
                        axis.set_ylabel(label)
                    if row == 0:
                        first_date = pd.Timestamp(household_history.start.iloc[0]).strftime("%d %b %Y").lstrip("0")
                        axis.set_title(f"{home} · observations start {first_date}", fontweight="bold", loc="left", pad=10)
                    if row == 2:
                        axis.set_xlabel("Calendar days of initial history")
                        axis.set_ylim(9, 24.5)
                    else:
                        axis.set_ylim(bottom=0)
            figure.suptitle("Household profiles as observations accumulate", x=0.08, y=0.98, ha="left", fontsize=17, fontweight="bold")
            figure.subplots_adjust(left=0.12, right=0.975, top=0.88, bottom=0.08, hspace=0.26, wspace=0.12)
            figure.savefig(figure_path, dpi=220, facecolor="white")
            plt.close(figure)

    calibration_figure = FIGURES_FOLDER / "F12_calibration_profiles.png"
    plot_calibration_profiles(calibration_results["fits"], calibration_figure)
    display(Image(filename=str(calibration_figure)))

In [ ]:
def analyse_h1_settings(settings_input_folder, settings_output_folder):
    from pathlib import Path
    import hashlib
    import json
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display
    INPUTS = Path(settings_input_folder)
    OUTPUTS = Path(settings_output_folder)
    OUTPUTS.mkdir(parents=True, exist_ok=True)
    validation_results = []

    def validate(name, condition):
        passed = bool(condition)
        validation_results.append({'rule': name, 'passed': passed})
        if not passed:
            raise AssertionError(name)

    def save_table(frame, name):
        frame.to_csv(OUTPUTS / name, index=False)
        return frame
    manifest = json.loads((INPUTS / 'manifest.json').read_text())
    for item in manifest['files']:
        actual = hashlib.sha256((INPUTS / item['file']).read_bytes()).hexdigest()
        validate('Input hash: ' + item['file'], actual == item['sha256'])
    metadata = json.loads((INPUTS / 'observation_metadata.json').read_text())
    data = pd.read_csv(INPUTS / 'daily_settings.csv', parse_dates=['date'])
    regimes = pd.read_csv(INPUTS / 'pump_helper_regimes.csv')
    coverage = pd.read_csv(INPUTS / 'pipe_coverage_by_pump_regime.csv')
    target_changes = pd.read_csv(INPUTS / 'thermostat_target_changes.csv')
    validate('Eligible dates are unique', data.date.is_unique)
    validate('Prepared daily observations contain 238 dates', len(data) == 238)
    validate('Gas remains nonnegative and observed', data.gas_kwh.notna().all() and data.gas_kwh.ge(0).all())

    def assign_pump_periods(frame, periods):
        result = frame[['date']].copy()
        result['start_utc'] = result.date.dt.tz_localize('Europe/London').dt.tz_convert('UTC')
        result['end_utc'] = (result.date + pd.Timedelta(days=1)).dt.tz_localize('Europe/London').dt.tz_convert('UTC')
        result['pump_setting'] = np.nan
        for period in periods.loc[periods.regime.ne('initial_helper_value')].itertuples():
            start = pd.Timestamp(period.start_utc)
            end = pd.Timestamp(period.end_utc)
            within = result.start_utc.ge(start) & result.end_utc.le(end)
            validate('Pump periods do not overlap: ' + period.regime, result.loc[within, 'pump_setting'].isna().all())
            result.loc[within, 'pump_setting'] = period.pump_setting
        return result
    assigned = assign_pump_periods(data, regimes)
    validate('Pump assignment matches prepared daily table', np.allclose(assigned.pump_setting, data.pump_setting, equal_nan=True))
    change_dates = pd.to_datetime(['2025-12-15', '2026-01-11', '2026-01-30'])
    validate('Pump change dates are unassigned', data.loc[data.date.isin(change_dates), 'pump_setting'].isna().all())
    validate('Pre-helper days are unassigned', data.loc[data.date.lt('2025-12-15'), 'pump_setting'].isna().all())
    validate('Days beyond the final full confirmation are unassigned', data.loc[data.date.ge('2026-08-07'), 'pump_setting'].isna().all())
    validate('Whole helper days total 223', int(data.pump_setting.notna().sum()) == 223)

    def pump_summary(frame):
        fields = ['gas_kwh', 'outside_mean_c', 'hdd_15_5', 'target_mean_c_20h', 'thermostat_air_mean_c_20h']
        rows = []
        selections = {'all_confirmed_days': frame.pump_setting.notna(), 'winter_dec16_feb28': frame.date.between('2025-12-16', '2026-02-28') & frame.pump_setting.notna()}
        for sample, mask in selections.items():
            for setting, group in frame.loc[mask].groupby('pump_setting'):
                row = {'sample': sample, 'pump_setting': setting, 'days': len(group), 'first_day': group.date.min().strftime('%Y-%m-%d'), 'last_day': group.date.max().strftime('%Y-%m-%d'), 'thermostat_days_ge20h': int(group.thermostat_eligible_20h.sum())}
                for field in fields:
                    values = group[field].dropna()
                    row[field + '_n'] = len(values)
                    for statistic in ['mean', 'median', 'min', 'max']:
                        row[field + '_' + statistic] = getattr(values, statistic)()
                rows.append(row)
        return pd.DataFrame(rows)
    pump_results = save_table(pump_summary(data), 'pump_descriptive_summary.csv')
    columns = ['pump_setting', 'days', 'thermostat_days_ge20h', 'gas_kwh_mean', 'outside_mean_c_mean', 'target_mean_c_20h_mean', 'thermostat_air_mean_c_20h_mean']
    change_times = pd.to_datetime(target_changes.target_timestamp_utc, utc=True)
    validate('Target-change timestamps are unique', change_times.is_unique)
    validate('Observed target-change count matches source audit', len(target_changes) == metadata['observed_target_change_count'] == 587)
    validate('Every change differs from its preceding numeric target', target_changes.target_c.ne(target_changes.previous_target_c).all())
    validate('Target changes span 134 local dates', change_times.dt.tz_convert('Europe/London').dt.date.nunique() == 134)
    start_local = data.date.dt.tz_localize('Europe/London')
    end_local = (data.date + pd.Timedelta(days=1)).dt.tz_localize('Europe/London')
    data['expected_local_hours'] = (end_local - start_local).dt.total_seconds() / 3600
    data['eligible_20h'] = data.available_hourly_statistics.ge(20) & data.target_mean_c.notna() & data.thermostat_air_mean_c.notna()
    data['complete_thermostat_day'] = data.available_hourly_statistics.eq(data.expected_local_hours) & data.eligible_20h
    validate('Thermostat source supplies 150 eligible dates', int(data.eligible_20h.sum()) == 150)
    validate('Complete thermostat sample contains 147 dates', int(data.complete_thermostat_day.sum()) == 147)
    validate('No target is filled after the outage', data.loc[data.date.gt('2026-05-24'), 'target_mean_c'].isna().all())
    samples = {'at_least_20_thermostat_hours': data.loc[data.eligible_20h].copy(), 'complete_thermostat_day': data.loc[data.complete_thermostat_day].copy()}
    excluded_complete = save_table(data.loc[data.eligible_20h & ~data.complete_thermostat_day, ['date', 'available_hourly_statistics', 'expected_local_hours', 'weather_hours']], 'complete_day_exclusions.csv')
    save_table(samples['at_least_20_thermostat_hours'], 'thermostat_150_day_sample.csv')
    weather = pd.read_csv(INPUTS / 'hourly_weather.csv')
    weather['timestamp'] = pd.to_datetime(weather.timestamp_utc, utc=True)
    weather['date'] = weather.timestamp.dt.tz_convert('Europe/London').dt.tz_localize(None).dt.normalize()
    validate('Hourly weather timestamps are unique', weather.timestamp.is_unique)
    for reference in [12.0, 15.5, 18.0]:
        field = f'hdd_{reference:g}'
        hourly_shortfall = np.maximum(reference - weather.temperature_c, 0)
        by_date = pd.DataFrame({'date': weather.date, 'shortfall': hourly_shortfall}).groupby('date').shortfall.mean()
        data[field] = data.date.map(by_date)
    validate('Recalculated 15.5C HDD matches prepared daily input', np.allclose(data['hdd_15.5'], data.hdd_15_5, rtol=1e-12, atol=1e-10))
    samples = {'at_least_20_thermostat_hours': data.loc[data.eligible_20h].copy(), 'complete_thermostat_day': data.loc[data.complete_thermostat_day].copy()}
    save_table(data, 'daily_settings_with_fixed_weather_features.csv')

    def design_matrix(frame, fields):
        return np.column_stack([np.ones(len(frame)), frame[fields].to_numpy()])

    def fit_model(training, fields):
        return np.linalg.lstsq(design_matrix(training, fields), training.gas_kwh.to_numpy(), rcond=None)[0]

    def estimate(frame, fields, coefficients):
        return design_matrix(frame, fields) @ coefficients

    def month_parts(frame, month):
        start = pd.Timestamp(month + '-01')
        end = start + pd.offsets.MonthBegin(1)
        return (frame.loc[frame.date.lt(start)].copy(), frame.loc[frame.date.ge(start) & frame.date.lt(end)].copy())

    def evaluate_models(sample_frames):
        scores, predictions, fitted = ([], [], {})
        for sample, frame in sample_frames.items():
            for reference in [12.0, 15.5, 18.0]:
                weather_field = f'hdd_{reference:g}'
                for month in ['2026-03', '2026-04', '2026-05']:
                    training, testing = month_parts(frame, month)
                    for model, fields in [('Weather', [weather_field]), ('Weather + target', [weather_field, 'target_mean_c'])]:
                        coefficients = fit_model(training, fields)
                        values = estimate(testing, fields, coefficients)
                        errors = values - testing.gas_kwh.to_numpy()
                        key = (sample, reference, month, model)
                        fitted[key] = {'fields': fields, 'coefficients': coefficients, 'training': training, 'testing': testing}
                        scores.append({'sample': sample, 'reference_c': reference, 'month': month, 'model': model, 'train_days': len(training), 'test_days': len(testing), 'mae_kwh': np.mean(abs(errors)), 'bias_kwh': np.mean(errors), 'intercept': coefficients[0], 'hdd_coefficient': coefficients[1], 'target_coefficient': coefficients[2] if len(coefficients) > 2 else np.nan, 'negative_estimate_days': int((values < 0).sum()), 'lowest_estimate_kwh': values.min()})
                        predictions.extend(({'sample': sample, 'reference_c': reference, 'month': month, 'model': model, 'date': date, 'actual_kwh': actual, 'estimated_kwh': value} for date, actual, value in zip(testing.date, testing.gas_kwh, values)))
        return (pd.DataFrame(scores), pd.DataFrame(predictions), fitted)
    scores, predictions, fitted_models = evaluate_models(samples)
    save_table(scores, 'model_months.csv')
    save_table(predictions, 'model_predictions.csv')
    primary = scores.loc[scores['sample'].eq('at_least_20_thermostat_hours') & scores.reference_c.eq(15.5)]
    with plt.rc_context({'font.family': 'DejaVu Sans', 'font.size': 10, 'text.color': '#303030', 'axes.labelcolor': '#303030', 'xtick.color': '#303030', 'ytick.color': '#303030', 'axes.spines.top': False, 'axes.spines.right': False}):
        figure, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharex=True, sharey=True)
        for axis, (sample, title) in zip(axes, [('at_least_20_thermostat_hours', 'At least 20 thermostat hours'), ('complete_thermostat_day', 'Complete thermostat days')]):
            rows = scores.loc[scores['sample'].eq(sample) & scores.reference_c.eq(15.5)]
            for model, colour, marker, offset in [('Weather', '#404040', 'o', -0.08), ('Weather + target', '#777777', 's', 0.08)]:
                selected = rows.loc[rows.model.eq(model)].sort_values('month')
                axis.scatter(selected.mae_kwh, np.arange(3) + offset, color=colour, marker=marker, s=48, label=model)
            axis.set_yticks(range(3), ['March', 'April', 'May'])
            axis.set_title(title, loc='left', weight='bold')
            axis.set_xlabel('Mean absolute error (kWh/day)')
            axis.set_xlim(left=0)
            axis.grid(axis='x', color='#DEDEDE', linewidth=0.7)
            axis.set_axisbelow(True)
        axes[0].invert_yaxis()
        axes[1].legend(frameon=False, loc='lower right')
        figure.tight_layout()
        figure.savefig(OUTPUTS / 'monthly_model_errors.png', dpi=180, facecolor='white')
        plt.close(figure)
    negative = save_table(predictions.loc[predictions.estimated_kwh.lt(0)].copy(), 'negative_estimates.csv')
    comparison = scores.pivot(index=['sample', 'reference_c', 'month'], columns='model', values='mae_kwh').reset_index()
    comparison['target_gain_kwh'] = comparison['Weather'] - comparison['Weather + target']
    save_table(comparison, 'model_gain_sensitivity.csv')
    monthly_context = data.loc[data.eligible_20h].assign(month=lambda frame: frame.date.dt.strftime('%Y-%m')).groupby('month').agg(days=('date', 'size'), mean_target_c=('target_mean_c', 'mean'), mean_outdoor_c=('outside_mean_c', 'mean'), mean_gas_kwh=('gas_kwh', 'mean'), mean_thermostat_air_c=('thermostat_air_mean_c', 'mean')).reset_index()
    save_table(monthly_context, 'monthly_context.csv')
    for key, model in fitted_models.items():
        sample, reference, month, name = key
        label = f'{sample}; {reference:g}C; {month}; {name}'
        fields = model['fields']
        training, testing = (model['training'], model['testing'])
        coefficients = model['coefficients']
        baseline = estimate(testing, fields, coefficients)
        validate('Training precedes assessment: ' + label, training.date.max() < testing.date.min())
        validate('Training and assessment dates are unique: ' + label, training.date.is_unique and testing.date.is_unique)
        changed = samples[sample].copy()
        changed.loc[changed.date.isin(testing.date), 'gas_kwh'] += 1000
        changed_training, changed_testing = month_parts(changed, month)
        changed_coefficients = fit_model(changed_training, fields)
        validate('Later gas does not change the fit: ' + label, np.array_equal(coefficients, changed_coefficients))
        validate('Later gas does not change estimates: ' + label, np.array_equal(baseline, estimate(changed_testing, fields, changed_coefficients)))
        validate('Later gas mutation actually changed outcomes: ' + label, np.allclose(changed_testing.gas_kwh - testing.gas_kwh, 1000))
        future_target = samples[sample].copy()
        future_target.loc[future_target.date.ge(pd.Timestamp(month + '-01')), 'target_mean_c'] += 5
        earlier, later = month_parts(future_target, month)
        future_coefficients = fit_model(earlier, fields)
        validate('Future target does not change earlier fit: ' + label, np.array_equal(coefficients, future_coefficients))
        if name == 'Weather + target':
            validate('Target input positive control: ' + label, not np.allclose(baseline, estimate(later, fields, future_coefficients)))
        else:
            validate('Unused target does not change weather-only estimates: ' + label, np.array_equal(baseline, estimate(later, fields, future_coefficients)))
        shifted_training = training.copy()
        shifted_training['gas_kwh'] += 5
        shifted_coefficients = fit_model(shifted_training, fields)
        validate('Training-gas positive control: ' + label, np.allclose(estimate(testing, fields, shifted_coefficients) - baseline, 5, rtol=1e-10, atol=1e-09))
    for key, rows in predictions.groupby(['sample', 'reference_c', 'month']):
        weather_dates = rows.loc[rows.model.eq('Weather'), 'date'].reset_index(drop=True)
        target_dates = rows.loc[rows.model.eq('Weather + target'), 'date'].reset_index(drop=True)
        validate('Methods use identical unique dates: ' + str(key), weather_dates.equals(target_dates) and weather_dates.is_unique)
    expected_pump = pd.read_csv(INPUTS / 'expected_pump_summary.csv')
    pd.testing.assert_frame_equal(pump_results, expected_pump, check_dtype=False, rtol=1e-12, atol=1e-10)
    validate('Pump descriptions match independent prepared reference', True)
    expected_models = pd.read_csv(INPUTS / 'expected_primary_model_scores.csv')
    comparison_fields = ['reference_c', 'month', 'model', 'train_days', 'test_days', 'mae_kwh', 'bias_kwh', 'target_coefficient']
    actual_models = scores.loc[scores['sample'].eq('at_least_20_thermostat_hours'), comparison_fields].reset_index(drop=True)
    pd.testing.assert_frame_equal(actual_models, expected_models[comparison_fields], check_dtype=False, rtol=1e-12, atol=1e-10)
    validate('Primary model scores match independent earlier calculation', True)
    validate('Setting 3 has no valid paired pipe statistics', coverage.loc[coverage.regime.eq('setting_3'), 'paired_numeric_nonfrozen_statistics_hours'].eq(0).all())
    validate('All declared variants preserve the March reversal', comparison.loc[comparison.month.eq('2026-03'), 'target_gain_kwh'].lt(0).all())
    validate('Both later months improve across declared variants', comparison.loc[comparison.month.isin(['2026-04', '2026-05']), 'target_gain_kwh'].gt(0).all())
    validation_frame = pd.DataFrame(validation_results)
    save_table(validation_frame, 'validation_rules.csv')
    validation_report = {'status': 'passed', 'rules_passed': int(validation_frame.passed.sum()), 'rules': validation_results, 'scope': 'Prepared-input identity, temporal separation, input mutations, matched dates and independent numerical reconciliation.', 'limitations': ['Does not verify original collection, sensor accuracy or physical actuation.', 'Does not remove earlier exploratory method selection or seasonal confounding.', 'Actual target-day weather and targets make the evaluation retrospective.']}
    (OUTPUTS / 'validation.json').write_text(json.dumps(validation_report, indent=2) + '\n')
    summary = {'status': 'exploratory_not_promoted', 'eligible_gas_weather_dates': len(data), 'confirmed_pump_days': int(data.pump_setting.notna().sum()), 'winter_pump_days': int(data.winter_comparison.sum()), 'thermostat_days_20h': len(samples['at_least_20_thermostat_hours']), 'complete_thermostat_days': len(samples['complete_thermostat_day']), 'observed_target_changes': len(target_changes), 'target_change_dates': 134, 'primary_monthly_scores': json.loads(primary.to_json(orient='records')), 'primary_test_dates': int(predictions.loc[predictions['sample'].eq('at_least_20_thermostat_hours') & predictions.reference_c.eq(15.5), 'date'].nunique()), 'complete_test_dates': int(predictions.loc[predictions['sample'].eq('complete_thermostat_day') & predictions.reference_c.eq(15.5), 'date'].nunique()), 'model_variants_reported': len(scores), 'validation_rules_passed': len(validation_results), 'promotion_decision': 'Do not promote these regressions to the main model or settings advice.', 'reasons': ['Adding target worsens March while improving April and May.', 'Unconstrained models produce negative May gas estimates.', 'Target, season and household behaviour are not independently varied.', 'Pump helper labels are not measured speed and setting 3 lacks paired pipe evidence.', 'A commanded boiler flow-temperature history is unavailable.']}
    (OUTPUTS / 'summary.json').write_text(json.dumps(summary, indent=2) + '\n')
    return {'daily_settings': data, 'pump_summary': pump_results, 'samples': samples, 'monthly_scores': scores, 'predictions': predictions, 'comparison': comparison, 'monthly_context': monthly_context, 'summary': summary, 'validation': validation_report}


In [ ]:
if reproduce_study:
    h1_settings_analysis = analyse_h1_settings(
        ANALYSIS_FOLDER / "supplementary/h1_settings/inputs",
        ANALYSIS_FOLDER / "outputs/h1_settings_models"
    )

In [ ]:
if reproduce_study:
    settings_folder = ANALYSIS_FOLDER / "supplementary/h1_settings"
    settings_input_folder = settings_folder / "inputs"
    settings_manifest = json.loads((settings_input_folder / "manifest.json").read_text())
    for item in settings_manifest["files"]:
        assert hashlib.sha256((settings_input_folder / item["file"]).read_bytes()).hexdigest() == item["sha256"]
    h1_daily_settings = pd.read_csv(settings_input_folder / "daily_settings.csv", parse_dates=["date"])
    settings_gas = h1_daily_settings.set_index("date").gas_kwh
    pd.testing.assert_series_equal(settings_gas, household_inputs["H1"]["gas"], check_names=False)
    assert np.allclose(h1_daily_settings.hdd_15_5, household_inputs["H1"]["degree_days"].loc[h1_daily_settings.date, 15.5])
    assert h1_daily_settings.date.is_unique
    thermostat_days = h1_daily_settings.loc[h1_daily_settings.available_hourly_statistics.ge(20)]
    winter_settings = h1_daily_settings.loc[h1_daily_settings.pump_eligible & h1_daily_settings.winter_comparison]
    pump_setting_summary = winter_settings.groupby("pump_setting").agg(
        days=("date", "size"), gas_kwh_day=("gas_kwh", "mean"),
        outdoor_c=("outside_mean_c", "mean"), target_c=("target_mean_c", "mean")
    ).reindex([2, 3, 1])
    assert len(thermostat_days) == 150 and pump_setting_summary.days.tolist() == [26, 18, 23]
    assert thermostat_days.date.max() == pd.Timestamp("2026-05-24")
    settings_summary = h1_settings_analysis["summary"]
    assert settings_summary["thermostat_days_20h"] == len(thermostat_days)
    assert settings_summary["winter_pump_days"] == len(winter_settings)
    settings_output_folder = ANALYSIS_FOLDER / "outputs/h1_settings"
    settings_output_folder.mkdir(parents=True, exist_ok=True)
    h1_daily_settings.to_csv(settings_output_folder / "daily_context.csv", index=False)
    pump_setting_summary.to_csv(settings_output_folder / "winter_pump_summary.csv")
    settings_validation = {
        "status": "passed", "input_files_verified": len(settings_manifest["files"]),
        "gas_and_hdd_match_main_analysis": True, "thermostat_days": len(thermostat_days),
        "winter_pump_days": len(winter_settings), "thermostat_models_recalculated": True,
        "scope": "Prepared inputs, matching dates, sample counts and recalculated thermostat models. No physical verification."
    }
    (ANALYSIS_FOLDER / "checks/h1_settings_summary.json").write_text(json.dumps(settings_validation, indent=2) + "\n");

In [ ]:
if reproduce_study:
    def plot_h1_settings(daily_settings, figure_path):
        thermostat = daily_settings.loc[daily_settings.available_hourly_statistics.ge(20)].set_index("date")
        calendar = pd.date_range(thermostat.index.min(), thermostat.index.max(), freq="D")
        temperatures = thermostat.reindex(calendar)
        winter_settings = daily_settings.loc[daily_settings.winter_comparison & daily_settings.pump_eligible]
        with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 9,
                             "text.color": "#303030", "axes.labelcolor": "#303030",
                             "xtick.color": "#303030", "ytick.color": "#303030"}):
            figure, axes = plt.subplots(2, 1, figsize=(7.2, 5.9))
            axes[0].plot(temperatures.index, temperatures.thermostat_air_mean_c,
                         color="#404040", linewidth=1.3, label="Air at thermostat")
            axes[0].plot(temperatures.index, temperatures.target_mean_c,
                         color="#777777", linestyle="--", linewidth=1.2, label="Logged target")
            axes[0].set_title("A  Target and air at the thermostat", loc="left", weight="bold", pad=10)
            axes[0].set_ylabel("Daily mean (°C)")
            axes[0].set_ylim(5, 26)
            axes[0].set_yticks([7, 12, 17, 22])
            axes[0].legend(frameon=False, ncol=2, loc="upper left", fontsize=8)
            axes[0].xaxis.set_major_locator(mdates.MonthLocator())
            axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
            styles = [(2, "s", "#999999", "#666666"), (3, "o", "white", "#555555"),
                      (1, "^", "#404040", "#404040")]
            for setting, marker, fill_colour, edge_colour in styles:
                setting_days = winter_settings.loc[winter_settings.pump_setting.eq(setting)]
                axes[1].scatter(setting_days.outside_mean_c, setting_days.gas_kwh, s=27, marker=marker,
                                facecolors=fill_colour, edgecolors=edge_colour, linewidths=.8,
                                label=f"Setting {setting}")
            axes[1].set_title("B  Gas use across recorded pump settings", loc="left", weight="bold", pad=10)
            axes[1].set_xlabel("Daily mean outdoor temperature (°C)")
            axes[1].set_ylabel("Daily gas (kWh)")
            axes[1].set_ylim(bottom=0)
            axes[1].legend(frameon=False, ncol=3, loc="upper right", fontsize=8)
            for axis in axes:
                axis.grid(axis="y", color="#DEDEDE", linewidth=.6)
                axis.set_axisbelow(True)
                axis.spines[["top", "right"]].set_visible(False)
            figure.tight_layout(h_pad=2.2)
            figure.savefig(figure_path, dpi=240, bbox_inches="tight", facecolor="white")
        return figure


    settings_figure_path = FIGURES_FOLDER / "F13_h1_settings.png"
    settings_figure = plot_h1_settings(h1_daily_settings, settings_figure_path)
    plt.close(settings_figure)
    display(Image(filename=str(settings_figure_path)))

In [ ]:
def validate_observation_data():
    validation_results = []

    def record_validation(rule_name, passed, detail):
        validation_results.append({"check": rule_name, "passed": bool(passed), "detail": detail})

    manifest = json.loads((OBSERVATION_INPUTS / "input_manifest.json").read_text())
    for source in manifest["inputs"]:
        digest = hashlib.sha256((OBSERVATION_INPUTS / source["file"]).read_bytes()).hexdigest()
        record_validation("Input integrity: " + source["file"], digest == source["sha256"], "SHA-256 matches the prepared-input manifest.")
    coverage, pipes, gas = load_observations()
    record_validation("Unique availability cells", len(coverage) == 100 and not coverage.duplicated(["home", "channel", "month"]).any(), "100 distinct home-channel-month cells.")
    record_validation("Coverage count accounting", (coverage["retained_intervals"] == coverage["recorded_intervals"] - coverage["excluded_known_frozen_intervals"]).all(), "Retained = recorded minus the documented exclusion.")
    record_validation("Coverage count bounds", coverage["retained_intervals"].between(0, coverage["expected_intervals"]).all(), "No negative counts or counts above their denominator.")
    calculated_coverage = 100 * coverage["retained_intervals"] / coverage["expected_intervals"].replace(0, np.nan)
    record_validation("Percentages match counts", np.allclose(calculated_coverage, coverage["coverage_percent"], equal_nan=True), "Every figure percentage recomputed from its numerator and denominator.")
    interval_count_matches = []
    for row in coverage.itertuples():
        month_start = pd.Timestamp(row.month, tz="UTC")
        month_end = month_start + pd.offsets.MonthBegin(1)
        start = max(month_start, pd.Timestamp(row.window_start_utc))
        end = min(month_end, pd.Timestamp(row.window_end_exclusive_utc))
        expected_intervals = len(pd.date_range(start, end, freq=f"{row.interval_minutes}min", inclusive="left")) if start < end else 0
        interval_count_matches.append(expected_intervals == row.expected_intervals)
    record_validation("Independent window denominators", all(interval_count_matches), "All 100 expected counts rebuilt with a UTC interval grid.")
    frozen_observations = coverage.loc[coverage["excluded_known_frozen_intervals"].gt(0)]
    record_validation("Known frozen episode retained in evidence", len(frozen_observations) == 1 and frozen_observations.iloc[0]["home"] == "H1" and frozen_observations.iloc[0]["channel"] == "Flow pipe" and frozen_observations.iloc[0]["month"] == "2026-01" and frozen_observations.iloc[0]["excluded_known_frozen_intervals"] == 428, "Exactly 428 excluded H1 January flow hours.")
    start, end = [pd.Timestamp(value) for value in manifest["trace_window_utc"]]
    for channel, pipe_observations in pipes.items():
        expected_rows = next(item["rows"] for item in manifest["inputs"] if item["file"] == f"h2_{channel}_selected.csv")
        record_validation("Observed trace rows: " + channel, len(pipe_observations) == expected_rows and pipe_observations["timestamp"].is_unique and pipe_observations["timestamp"].is_monotonic_increasing and pipe_observations["timestamp"].ge(start).all() and pipe_observations["timestamp"].lt(end).all() and np.isfinite(pipe_observations["temperature_c"]).all(), "Timestamp order, window, finite values and row count match the selected observations.")
        exported = pd.read_csv(OBSERVATION_OUTPUTS / f"F02_{channel}_observations.csv")
        record_validation("No temperature interpolation: " + channel, np.array_equal(exported["temperature_c"], pipe_observations["temperature_c"]) and np.array_equal(pd.to_datetime(exported["timestamp"], utc=True), pipe_observations["timestamp"]), "Every exported plotted value and timestamp equals its input observation.")
    valid_gas_intervals = (gas["interval_end"] - gas["interval_start"]).eq(pd.Timedelta(minutes=30)).all()
    adjacent_intervals = np.array_equal(gas["interval_end"].iloc[:-1], gas["interval_start"].iloc[1:])
    record_validation("Native gas bins and full selected window", len(gas) == 8 and valid_gas_intervals and adjacent_intervals and gas["interval_start"].min() == start and gas["interval_end"].max() == end and gas["kwh"].ge(0).all(), "Eight adjacent original half-hours; no gas interpolation or inserted zero bins.")
    zero_gas_intervals = gas.loc[gas["interval_start"].ge(pd.Timestamp("2026-01-02 12:00", tz="UTC")) & gas["interval_end"].le(pd.Timestamp("2026-01-02 13:00", tz="UTC"))]
    record_validation("Annotated zero-gas interval", len(zero_gas_intervals) == 2 and zero_gas_intervals["kwh"].eq(0).all(), "Both retained bins between 12:00 and 13:00 report zero kWh.")
    totals = summarise_availability(coverage)
    record_validation("Summary preserves underlying counts", totals["retained_intervals"].sum() == coverage["retained_intervals"].sum(), "Home-channel summaries reconcile to all monthly retained intervals.")
    summary = {"status": "passed" if all(row["passed"] for row in validation_results) else "failed", "checks": validation_results, "count": len(validation_results), "scope": "Prepared-input integrity, availability accounting and exact observation exports. Data validation does not establish sensor accuracy, original household extraction, device uptime or burner firing."}
    (ANALYSIS_FOLDER / "checks/observation_results.json").write_text(json.dumps(summary, indent=2) + "\n")
    if summary["status"] != "passed":
        raise AssertionError([row for row in validation_results if not row["passed"]])
    return summary

In [ ]:
if reproduce_study:
    observation_validation = validate_observation_data()

In [ ]:
def validate_daily_data():
    daily_input_folder = ANALYSIS_FOLDER / "inputs/daily"
    household_inputs, coverage = load_daily_inputs(daily_input_folder)
    monthly_results, gas_estimates = compare_daily_estimates(household_inputs)
    passed_rules = []

    def validate_rule(rule_name, condition):
        if not condition:
            raise AssertionError(rule_name)
        passed_rules.append(rule_name)

    for item in json.loads((daily_input_folder / "input_manifest.json").read_text()):
        validate_rule(f"hash:{item['file']}", hashlib.sha256((daily_input_folder / item["file"]).read_bytes()).hexdigest() == item["sha256"])
    for home, expected in (("H1", 238), ("H2", 226)):
        household_data = household_inputs[home]
        validate_rule(f"aligned_unique_days:{home}", household_data["gas"].index.equals(household_data["degree_days"].index)
              and household_data["gas"].index.is_unique and len(household_data["gas"]) == expected)
    expected_mae = {
        ("H1", COMPARISON_METHODS[0]): 6.655136386038805,
        ("H1", COMPARISON_METHODS[2]): 6.786568394000654,
        ("H2", COMPARISON_METHODS[0]): 4.485038756853071,
        ("H2", COMPARISON_METHODS[2]): 5.775281746994989,
    }
    summary = monthly_results.groupby(["home", "model"]).mae_kwh_day.mean()
    for key, expected in expected_mae.items():
        validate_rule(f"revised_mae:{key[0]}:{key[1]}", np.isclose(summary.loc[key], expected, rtol=0, atol=1e-10))
    for home, household_data in household_inputs.items():
        dates = household_data["gas"].index
        for month in monthly_results.loc[monthly_results.home.eq(home), "month"].unique():
            period = pd.Period(month)
            training_days = dates < period.start_time
            assessment_days = dates.to_period("M") == period
            validate_rule(f"disjoint_ordered_split:{home}:{month}", dates[training_days].max() < dates[assessment_days].min())
            selected = gas_estimates.home.eq(home) & gas_estimates.month.eq(month)
            original_estimates = gas_estimates.loc[selected].reset_index(drop=True)
            modified_households = copy.deepcopy(household_inputs)
            modified_households[home]["gas"].loc[assessment_days] += 1000
            _, changed_estimates = compare_daily_estimates(modified_households)
            changed_estimates = changed_estimates.loc[changed_estimates.home.eq(home) & changed_estimates.month.eq(month)].reset_index(drop=True)
            validate_rule(f"heldout_target_invariance:{home}:{month}", np.array_equal(original_estimates.predicted_kwh, changed_estimates.predicted_kwh))
            validate_rule(f"target_mutation_control:{home}:{month}", np.allclose(changed_estimates.actual_kwh - original_estimates.actual_kwh, 1000))
            original_profile = fit_gas_profile(household_data["gas"].loc[training_days], household_data["degree_days"].loc[training_days])
            changed_profile = fit_gas_profile(modified_households[home]["gas"].loc[training_days], modified_households[home]["degree_days"].loc[training_days])
            validate_rule(f"heldout_fit_invariance:{home}:{month}", original_profile == changed_profile)
            for order in ("C", "F"):
                reordered_degree_days = pd.DataFrame(np.array(household_data["degree_days"].loc[training_days], order=order),
                                       index=dates[training_days], columns=household_data["degree_days"].columns)
                validate_rule(f"array_layout_invariance:{home}:{month}:{order}",
                      original_profile == fit_gas_profile(household_data["gas"].loc[training_days].copy(), reordered_degree_days))
            temperature = household_data["temperature"].copy()
            later_dates = np.array(temperature.index.date) > period.end_time.date()
            if later_dates.any():
                for label, replacement in (("values", 500.0), ("missingness", np.nan)):
                    temperature.loc[later_dates] = replacement
                    changed_weather = calculate_degree_days(temperature)
                    validate_rule(f"future_weather_{label}_invariance:{home}:{month}",
                          household_data["degree_days"].loc[dates[training_days | assessment_days]].equals(
                              changed_weather.loc[dates[training_days | assessment_days]]))
            validate_rule(f"matched_method_dates:{home}:{month}", original_estimates.groupby("model").date.apply(tuple).nunique() == 1)
    index = pd.date_range("2026-01-01", periods=48, freq="h", tz="Europe/London")
    temperatures = pd.Series(np.tile([0.0, 20.0], 24), index=index)
    temperatures.iloc[24:29] = np.nan
    weather = calculate_degree_days(temperatures)
    validate_rule("clip_hourly_before_daily_mean", weather.loc[pd.Timestamp("2026-01-01"), 15.5] == 7.75)
    validate_rule("reject_fewer_than_20_weather_hours", list(weather.index) == [pd.Timestamp("2026-01-01")])
    temperatures = np.linspace(0.0, 5.0, 50)
    weather = pd.DataFrame(REFERENCE_TEMPERATURES[None, :] - temperatures[:, None], columns=REFERENCE_TEMPERATURES)
    example_profile = fit_gas_profile(pd.Series(31.0 - 2.0 * temperatures), weather)
    validate_rule("lowest_tied_grid_known_answer", example_profile.balance_temperature_c == 10.0
          and example_profile.tied_candidates == 23 and np.isclose(example_profile.slope_kwh_per_degree_day, 2.0)
          and np.isclose(example_profile.base_kwh_per_day, 11.0))
    for home, household_data in household_inputs.items():
        original_profile = fit_gas_profile(household_data["gas"], household_data["degree_days"])
        shifted_profile = fit_gas_profile(household_data["gas"] + 5.0, household_data["degree_days"])
        validate_rule(f"training_target_dependency:{home}", np.allclose(
            shifted_profile.estimate_gas(household_data["degree_days"]) - original_profile.estimate_gas(household_data["degree_days"]), 5.0))
    with tempfile.TemporaryDirectory() as temporary:
        copied_inputs = Path(temporary) / "inputs"
        shutil.copytree(daily_input_folder, copied_inputs)
        with (copied_inputs / "h1_daily_gas.csv").open("a") as handle:
            handle.write("\n")
        input_rejected = False
        try:
            load_daily_inputs(copied_inputs)
        except ValueError:
            input_rejected = True
        validate_rule("reject_changed_input", input_rejected)
    result = {
        "status": "passed", "checks_passed": len(passed_rules), "checks": passed_rules,
        "python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
        "matched_days": dict(zip(coverage.home, coverage.matched_days)),
        "monthly_splits": int(len(monthly_results[["home", "month"]].drop_duplicates())),
        "monthly_mae": [{"home": home, "model": model, "mae_kwh_day": float(value)}
                        for (home, model), value in summary.items()],
        "full_record_fits": {home: asdict(fit_gas_profile(household_data["gas"], household_data["degree_days"]))
                             for home, household_data in household_inputs.items()},
        "tie_rule": "Contiguous arrays; lowest grid candidate within np.isclose(loss, minimum_loss, rtol=1e-10, atol=1e-8).",
        "ambiguous_monthly_fits": monthly_results.loc[monthly_results.model.eq(COMPARISON_METHODS[0]) & monthly_results.tied_candidates.gt(1),
                                             ["home", "month", "tied_candidates", "tied_low_c", "tied_high_c"]].to_dict("records"),
        "limits": "Data validation covers the supplied extracts and derived daily values. It does not establish raw-collection accuracy, forecasting performance or physical efficiency, and it cannot create an untouched final test.",
    }
    (ANALYSIS_FOLDER / "checks/daily_results.json").write_text(json.dumps(result, indent=2) + "\n")
    return result

In [ ]:
if reproduce_study:
    daily_validation = validate_daily_data()

In [ ]:
def validate_weather_data():
    hourly_inputs = load_hourly_weather(ANALYSIS_FOLDER / "inputs/comparison")
    gas, weather = prepare_weather_inputs(hourly_inputs)
    monthly_results, gas_estimates = compare_weather_sources(gas, weather)
    summary = summarise_weather_results(gas_estimates)
    validation_results = []

    def record_validation(rule_name, passed, **evidence):
        validation_results.append({"test": rule_name, "passed": bool(passed), **evidence})
        if not passed:
            raise AssertionError(rule_name)

    record_validation("Unique hourly and daily keys", hourly_inputs.index.is_unique and gas.index.is_unique)
    record_validation("Matched observation counts", len(gas) == 226 and len(gas_estimates) == 314)
    reference_keys = gas_estimates.loc[gas_estimates.source.eq("Local outdoor"), ["date", "month"]]
    comparison_keys = gas_estimates.loc[gas_estimates.source.eq("ERA5"), ["date", "month"]]
    record_validation("Both sources score exactly the same dates", np.array_equal(reference_keys, comparison_keys))
    record_validation("Weather inputs have exactly the shared daily index",
           all(source_degree_days.index.equals(gas.index) for source_degree_days in weather.values()))

    missing_day_inputs = hourly_inputs.copy()
    missing_day = gas.index[10]
    missing_day_inputs.loc[missing_day_inputs.index.tz_localize(None).normalize() == missing_day, "gas_increment_kwh"] = np.nan
    missing_gas, _ = prepare_weather_inputs(missing_day_inputs)
    record_validation("A missing gas day is excluded, never filled with zero", missing_day not in missing_gas.index)

    for month, training_dates, assessment_dates in monthly_evaluation_dates(gas.index):
        record_validation(f"{month}: dates are disjoint and chronological",
               len(training_dates.intersection(assessment_dates)) == 0 and training_dates.max() < assessment_dates.min())
        changed_gas = gas.copy()
        changed_gas.loc[assessment_dates] = np.arange(len(assessment_dates)) + 1000.0
        changed_monthly_results, changed_estimates = compare_weather_sources(changed_gas, weather)
        for source, source_degree_days in weather.items():
            mask = gas_estimates.month.eq(month) & gas_estimates.source.eq(source)
            month_selection = monthly_results.month.eq(month) & monthly_results.source.eq(source)
            profile_fields = ["balance_temperature_c", "slope_kwh_per_degree_day", "base_kwh_per_day", "r_squared"]
            record_validation(f"{month} {source}: test targets cannot change this fold's fit or predictions",
                   monthly_results.loc[month_selection, profile_fields].equals(changed_monthly_results.loc[month_selection, profile_fields])
                   and gas_estimates.loc[mask, "predicted_kwh"].equals(changed_estimates.loc[mask, "predicted_kwh"]))

        next_month = pd.Timestamp(month).tz_localize("UTC") + pd.offsets.MonthBegin(1)
        if (hourly_inputs.index >= next_month).any():
            changed = hourly_inputs.copy()
            later_row_positions = np.flatnonzero(changed.index >= next_month)
            changed.iloc[later_row_positions, :] = 1000.0
            changed.iloc[later_row_positions[::2], :] = np.nan
            earlier_gas, earlier_weather = prepare_weather_inputs(changed)
            changed_monthly_results, changed_estimates = compare_weather_sources(earlier_gas, earlier_weather)
            retained = gas_estimates[gas_estimates.month.le(month)].reset_index(drop=True)
            repeated = changed_estimates[changed_estimates.month.le(month)].reset_index(drop=True)
            retained_monthly_results = monthly_results[monthly_results.month.le(month)].reset_index(drop=True)
            repeated_monthly_results = changed_monthly_results[changed_monthly_results.month.le(month)].reset_index(drop=True)
            record_validation(f"{month}: later input values and missingness cannot change earlier results",
                   retained.equals(repeated) and retained_monthly_results.equals(repeated_monthly_results))

    _, training_dates, assessment_dates = next(monthly_evaluation_dates(gas.index))
    first_test = assessment_dates.min().tz_localize("UTC")
    after_test = (assessment_dates.max() + pd.Timedelta(days=1)).tz_localize("UTC")
    changed = hourly_inputs.copy()
    test_hours = (changed.index >= first_test) & (changed.index < after_test)
    changed.loc[test_hours, ["local_outdoor_C", "era5_outdoor_C"]] += 20.0
    changed_gas, changed_weather = prepare_weather_inputs(changed)
    for source, source_degree_days in weather.items():
        original_profile = fit_gas_profile(gas.loc[training_dates], source_degree_days.loc[training_dates])
        changed_profile = fit_gas_profile(changed_gas.loc[training_dates], changed_weather[source].loc[training_dates])
        record_validation(f"{source}: target-day weather changes estimates, confirming retrospective input use",
               asdict(original_profile) == asdict(changed_profile)
               and not np.allclose(original_profile.estimate_gas(source_degree_days.loc[assessment_dates]),
                                   changed_profile.estimate_gas(changed_weather[source].loc[assessment_dates])))

    monthly_errors = gas_estimates.groupby(["source", "month"]).absolute_error_kwh.agg(["mean", "count"])
    for row in summary.itertuples():
        month_errors = monthly_errors.loc[row.source]
        weighted_error = np.average(month_errors["mean"], weights=month_errors["count"])
        record_validation(f"{row.source}: pooled MAE equals a day-weighted monthly average",
               np.isclose(weighted_error, row.mae_kwh_day, atol=1e-12))

    validation_summary = {
        "scope": "Prepared H1 hourly input, shared daily model and matched UTC weather comparison",
        "calendar": "UTC; distinct from local civil days in the main daily notebook",
        "passed": all(row["passed"] for row in validation_results), "checks": len(validation_results),
        "matched_days": len(gas), "prediction_rows": len(gas_estimates),
        "summary": summary.to_dict("records"), "results": validation_results,
        "input_sha256": {
            "inputs/comparison/h1_hourly_inputs.parquet": hashlib.sha256(
                (ANALYSIS_FOLDER / "inputs/comparison/h1_hourly_inputs.parquet").read_bytes()
            ).hexdigest()
        },
        "inherited_algorithm_source_sha256": {
            "comparison.py": "6b623c01c0ee01a1fdf5bfd1f5862d11fa72931ac6b853d9d1dc4ef0c83ee5ea",
            "daily.py": "1d3d3d78a7eed73290835c2bd53b44c94a67f2443b61b1d5f195f85d818ae358"
        },
        "limit": "These tests do not remove research-selection effects or turn observed-weather estimates into forecasts.",
    }
    (ANALYSIS_FOLDER / "checks/comparison_results.json").write_text(json.dumps(validation_summary, indent=2) + "\n")
    return validation_summary

In [ ]:
if reproduce_study:
    weather_validation = validate_weather_data()

In [ ]:
def validate_sensor_data():
    validation_results = []

    def record_validation(rule_name, passed):
        validation_results.append({"check": rule_name, "passed": bool(passed)})
        if not passed:
            raise AssertionError(rule_name)

    for item in json.loads((SENSOR_INPUTS / "input_manifest.json").read_text())["inputs"]:
        digest = hashlib.sha256((SENSOR_INPUTS / item["file"]).read_bytes()).hexdigest()
        record_validation("Input integrity: " + item["file"], digest == item["sha256"])
    minute_times = pd.date_range("2026-01-01", periods=60, freq="1min", tz="UTC")
    pipe_temperatures = {"flow": pd.Series(50.0, index=minute_times),
                "return": pd.Series(np.r_[np.full(30, 52.0), np.full(30, 47.0)], index=minute_times)}
    paired_half_hours = summarise_pipe_half_hours(pipe_temperatures)
    record_validation("Known signed gaps retain negative values", np.allclose(paired_half_hours.gap_mean_C, [-2, 3]))
    gas = pd.Series([0.0, 1.0], index=paired_half_hours.index, name="gas_kwh")
    selected = select_complete_intervals(paired_half_hours, gas)
    record_validation("Zero-gas rows remain eligible", len(selected) == 2 and selected.gas_kwh.iloc[0] == 0)
    missing_minute_example = {**pipe_temperatures, "return": pipe_temperatures["return"].drop(minute_times[1])}
    record_validation("One missing minute excludes a half-hour without filling",
           select_complete_intervals(summarise_pipe_half_hours(missing_minute_example), gas).index.equals(paired_half_hours.index[1:]))
    uneven_minute_example = {**pipe_temperatures, "flow": pd.concat([pipe_temperatures["flow"], pd.Series([60.0], index=[minute_times[0] + pd.Timedelta(seconds=30)])]).sort_index()}
    record_validation("Uneven observation counts receive equal minute weights", np.isclose(summarise_pipe_half_hours(uneven_minute_example).flow_mean_C.iloc[0], 50 + 5 / 30))
    changed = select_complete_intervals(paired_half_hours, gas + 10)
    record_validation("Gas values cannot change temperature features or matching", changed.drop(columns="gas_kwh").equals(selected.drop(columns="gas_kwh")))
    gas_interval_examples = pd.DataFrame({"interval_start": ["2026-01-01 00:00", "2026-03-29 01:00", "2026-01-01 01:00"],
                              "interval_end": ["2026-01-01 00:30", "2026-03-29 01:30", "2026-01-01 02:00"],
                              "kwh": [0.0, 1.0, 2.0]})
    retained, excluded = prepare_gas_intervals(gas_interval_examples)
    record_validation("Invalid civil times and non-half-hours are rejected; zero is retained", len(retained) == 1 and excluded == 2 and retained.iloc[0] == 0)
    sample, summary = analyse_pipe_sensor_value()
    record_validation("Recorded sample and gas boundary reproduce", len(sample) == 76 and summary["UTC_dates"] == 32 and summary["zero_gas_bins"] == 0 and summary["excluded_invalid_gas_intervals"] == 1)
    record_validation("Independent prior mean-gap result reproduces", np.isclose(sample.gap_mean_C.corr(sample.gas_kwh), 0.9741117922216217, atol=1e-12))
    record_validation("Independent prior one-sensor result reproduces", np.isclose(sample.flow_mean_C.corr(sample.gas_kwh), 0.0038091440542470404, atol=1e-12))
    record_validation("Later absence of a qualifying sample remains visible", summary["later_full_clock_bins"] == 3448 and summary["later_complete_temperature_bins"] == 0 and summary["later_zero_gas_bins"] == 2734 and summary["later_positive_gas_bins"] == 693)
    exported_sample = pd.read_csv(SENSOR_OUTPUTS / "F07_matched_half_hours.csv", index_col=0, parse_dates=True)
    pd.testing.assert_frame_equal(sample, exported_sample, check_names=False, check_freq=False)
    record_validation("Plotted table preserves matched values and timestamps", True)
    result = {"status": "passed", "count": len(validation_results), "checks": validation_results,
              "scope": "Prepared-input integrity, known-answer pairing, negative differences, missingness, zero-gas eligibility, UTC validation, sample arithmetic and exports. Does not validate sensor installation, calibration, freshness, physical heat or general performance."}
    (ANALYSIS_FOLDER / "checks/sensor_value_results.json").write_text(json.dumps(result, indent=2) + "\n")
    return result

In [ ]:
if reproduce_study:
    sensor_validation = validate_sensor_data()

In [ ]:
if reproduce_study:
    validation_summary = pd.DataFrame([
        {"Group": "Observations", "Validation rules passed": observation_validation["count"]},
        {"Group": "Daily model", "Validation rules passed": daily_validation["checks_passed"]},
        {"Group": "Weather comparison", "Validation rules passed": weather_validation["checks"]},
        {"Group": "Sensor comparison", "Validation rules passed": sensor_validation["count"]},
    ])
    assert int(validation_summary["Validation rules passed"].sum()) == 184
    validation_summary.to_csv(ANALYSIS_FOLDER / "checks/check_summary.csv", index=False)
    display(HTML(validation_summary.to_html(index=False)))

In [ ]:
if reproduce_study:
    from dataclasses import asdict, dataclass
    from pathlib import Path
    import copy
    import hashlib
    import json
    import numpy as np
    import pandas as pd

    EARLIER_INPUTS = INPUTS_FOLDER / "earlier_experiments"
    EARLIER_OUTPUTS = ANALYSIS_FOLDER / "outputs/earlier_experiments"
    EARLIER_OUTPUTS.mkdir(parents=True, exist_ok=True);


In [ ]:
if reproduce_study:
    @dataclass(frozen=True)
    class earlier_event_EventConfig:
        forward_fill_minutes: int = 5
        quiet_gap_minutes: int = 60
        quiet_gap_tolerance_C: float = 2.0
        merge_minutes: int = 10
        minimum_minutes: int = 5
        final_rise_C_per_minute: float = 0.5
        slope_window_minutes: int = 5
        slope_rise: float = 0.15
        slope_fall: float = -0.15
        timezone: str = 'Europe/London'
        reference_start: str = '2025-12-08'
        reference_end: str = '2026-01-12 17:00'
        bootstrap_draws: int = 10000
        bootstrap_seed: int = 20260815

        @property
        def start(self):
            return pd.Timestamp(self.reference_start, tz='UTC')

        @property
        def end(self):
            return pd.Timestamp(self.reference_end, tz='UTC')

        @property
        def midpoint(self):
            return self.start + (self.end - self.start) / 2
    earlier_event_CONFIG = earlier_event_EventConfig()
    earlier_event_FFILL_LIMIT_MIN = earlier_event_CONFIG.forward_fill_minutes
    earlier_event_BENIGN_GAP_CAP_MIN = earlier_event_CONFIG.quiet_gap_minutes
    earlier_event_BENIGN_GAP_TOL_C = earlier_event_CONFIG.quiet_gap_tolerance_C
    earlier_event_MERGE_GAP_MIN = earlier_event_CONFIG.merge_minutes
    earlier_event_MIN_DUR_MIN = earlier_event_CONFIG.minimum_minutes
    earlier_event_RISE_TRIM = earlier_event_CONFIG.final_rise_C_per_minute

    def earlier_event_grid_1min(s, cap=earlier_event_BENIGN_GAP_CAP_MIN, tol=earlier_event_BENIGN_GAP_TOL_C, strict=False):
        g = s.resample('1min').last()
        filled = g.ffill(limit=earlier_event_FFILL_LIMIT_MIN)
        if strict:
            return filled
        isna = filled.isna()
        if isna.any():
            grp = (isna != isna.shift()).cumsum()
            interp = g.interpolate(method='time', limit_area='inside')
            for _, idxs in filled[isna].groupby(grp[isna]):
                i0, i1 = (idxs.index[0], idxs.index[-1])
                before, after = (s.loc[:i0], s.loc[i1:])
                if len(before) == 0 or len(after) == 0:
                    continue
                if (i1 - i0).total_seconds() / 60 <= cap and abs(before.iloc[-1] - after.iloc[0]) <= tol:
                    filled.loc[i0:i1] = interp.loc[i0:i1]
        return filled

    def earlier_event_flow_modes(vals, bw=1.0):
        hist, edges = np.histogram(vals, bins=np.arange(np.floor(vals.min()), np.ceil(vals.max()) + 1, bw))
        c = (edges[:-1] + edges[1:]) / 2
        smv = np.convolve(hist, np.ones(5) / 5, mode='same')
        return (float(c[np.argmax(np.where(c < 45, smv, 0))]), float(c[np.argmax(np.where(c > 60, smv, 0))]), c, smv)


In [ ]:
if reproduce_study:
    def earlier_event_detect_events(g, th_on, th_off, merge_min=earlier_event_MERGE_GAP_MIN, min_dur=earlier_event_MIN_DUR_MIN, rise=earlier_event_RISE_TRIM):
        evs, state, on = ([], False, None)
        for t, v in zip(g.index, g.values):
            if np.isnan(v):
                if state:
                    evs.append([on, t, True])
                    state = False
                continue
            if not state and v >= th_on:
                state, on = (True, t)
            elif state and v < th_off:
                evs.append([on, t, False])
                state = False
        if state:
            evs.append([on, g.index[-1], True])
        merged = []
        for st, en, ce in evs:
            if merged and (st - merged[-1][1]).total_seconds() <= merge_min * 60:
                merged[-1][1] = en
                merged[-1][2] = merged[-1][2] or ce
            else:
                merged.append([st, en, ce])
        out = []
        for st, en, ce in merged:
            if not ce:
                seg = g.loc[st:en].dropna()
                v = seg.values
                rises = np.where(np.diff(v) >= rise)[0]
                if len(rises):
                    j = rises[-1] + 1
                    while j < len(v) - 1 and v[j + 1] >= v[j]:
                        j += 1
                    en = seg.index[j]
            out.append([st, en, ce])
        ev = pd.DataFrame(out, columns=['start', 'end', 'censored'])
        ev['dur'] = (ev.end - ev.start).dt.total_seconds() / 60
        return ev[ev.dur >= min_dur].reset_index(drop=True)

    def earlier_event_events_from_binary(b, merge_min=earlier_event_MERGE_GAP_MIN, min_dur=earlier_event_MIN_DUR_MIN):
        ev, cur = ([], None)
        for t, v in b.items():
            if v == 'on' and cur is None:
                cur = t
            elif v == 'off' and cur is not None:
                ev.append((cur, t))
                cur = None
        rawe = pd.DataFrame(ev, columns=['start', 'end'])
        rawe['dur'] = (rawe.end - rawe.start).dt.total_seconds() / 60
        m = []
        for r in rawe.itertuples():
            if m and (r.start - m[-1][1]).total_seconds() <= merge_min * 60:
                m[-1][1] = r.end
            else:
                m.append([r.start, r.end])
        gt = pd.DataFrame(m, columns=['start', 'end'])
        gt['dur'] = (gt.end - gt.start).dt.total_seconds() / 60
        return (gt[gt.dur >= min_dur].reset_index(drop=True), rawe)

    def earlier_event_minute_pred(det, index):
        p = pd.Series(0.0, index=index)
        for d in det.itertuples():
            p.loc[d.start:d.end] = 1.0
        return p

    def earlier_event_minute_scores(pred, on_minutes):
        ok = on_minutes.notna()
        tp = int(((pred == 1) & (on_minutes == 1) & ok).sum())
        fp = int(((pred == 1) & (on_minutes == 0) & ok).sum())
        fn = int(((pred == 0) & (on_minutes == 1) & ok).sum())
        mP = tp / (tp + fp) if tp + fp else 0.0
        mR = tp / (tp + fn) if tp + fn else 0.0
        return dict(mP=mP, mR=mR, mF1=2 * mP * mR / (mP + mR) if mP + mR else 0.0, tp=tp, fp=fp, fn=fn)

    def earlier_event_score(det, gtev, on_minutes):
        dm = np.zeros(len(det), bool)
        gm = np.zeros(len(gtev), bool)
        onset, offset, mult = ([], [], [])
        for i, d in det.iterrows():
            ov = gtev[(gtev.end > d.start) & (gtev.start < d.end)]
            mult.append(len(ov))
            if len(ov):
                dm[i] = True
                gm[ov.index] = True
                onset.append((d.start - ov.iloc[0].start).total_seconds() / 60)
                offset.append((d.end - ov.iloc[-1].end).total_seconds() / 60)
        P = dm.mean() if len(det) else 0.0
        R = gm.mean() if len(gtev) else 0.0
        ms = earlier_event_minute_scores(earlier_event_minute_pred(det, on_minutes.index), on_minutes)
        return dict(P=P, R=R, F1=2 * P * R / (P + R) if P + R > 0 else 0.0, n_det=len(det), n_gt=len(gtev), onset=np.array(onset), offset=np.array(offset), det_matched=dm, gt_matched=gm, multiplicity=np.array(mult), **ms)


In [ ]:
if reproduce_study:
    def earlier_event_digest(path):
        return hashlib.sha256(Path(path).read_bytes()).hexdigest()

    def earlier_event_validate_inputs(folder):
        manifest = json.loads((folder / 'input_manifest.json').read_text())
        needed = {'h1_flow.parquet', 'h1_reference.parquet', 'h2_flow.parquet', 'h2_gas_intervals.parquet'}
        inputs = [item for item in manifest['inputs'] if item['file'] in needed]
        assert {item['file'] for item in inputs} == needed
        for item in inputs:
            assert earlier_event_digest(folder / item['file']) == item['sha256'], item['file']
        return pd.DataFrame(inputs)[['file', 'rows', 'role']]

    def earlier_event_load_temperature(folder, name):
        frame = pd.read_parquet(folder / name)
        assert list(frame.columns) == ['timestamp', 'value']
        result = frame.set_index('timestamp')['value'].sort_index()
        assert result.index.tz is not None
        assert result.notna().all()
        return result

    def earlier_event_prepare_signals(folder, config=earlier_event_CONFIG):
        h1 = earlier_event_load_temperature(folder, 'h1_flow.parquet').loc[config.start:config.end]
        h2 = earlier_event_load_temperature(folder, 'h2_flow.parquet')
        reference = pd.read_parquet(folder / 'h1_reference.parquet')
        assert list(reference.columns) == ['timestamp', 'state']
        reference = reference.set_index('timestamp')['state'].sort_index()
        reference = reference.loc[config.start:config.end]
        grids = {h: earlier_event_grid_1min(raw, config.quiet_gap_minutes, config.quiet_gap_tolerance_C) for h, raw in [('H1', h1), ('H2', h2)]}
        return (grids, reference)

    def earlier_event_tune_and_score(grids, reference, config=earlier_event_CONFIG):
        reference_events, raw_reference_events = earlier_event_events_from_binary(reference)
        reference_minutes = (reference == 'on').astype(float).resample('1min').last().ffill()
        reference_minutes = reference_minutes.reindex(grids['H1'].index)
        modes = {}
        for house, data in [('H1', grids['H1'].loc[:config.midpoint]), ('H2', grids['H2'])]:
            idle, warm, centres, frequency = earlier_event_flow_modes(data.dropna().to_numpy())
            modes[house] = {'idle': idle, 'warm': warm, 'centres': centres, 'frequency': frequency}
        training_reference = reference_events[reference_events.end <= config.midpoint].reset_index(drop=True)
        test_reference = reference_events[reference_events.start > config.midpoint].reset_index(drop=True)
        rows = []
        idle, warm = (modes['H1']['idle'], modes['H1']['warm'])
        for upper in [0.3, 0.4, 0.5, 0.6, 0.7]:
            for lower in [0.1, 0.2, 0.3, 0.4]:
                if lower >= upper:
                    continue
                events = earlier_event_detect_events(grids['H1'].loc[:config.midpoint], idle + upper * (warm - idle), idle + lower * (warm - idle))
                metrics = earlier_event_score(events, training_reference, reference_minutes.loc[:config.midpoint])
                rows.append({'upper_fraction': upper, 'lower_fraction': lower, 'event_f1': round(metrics['F1'], 3), 'minute_f1': round(metrics['mF1'], 3)})
        tuning = pd.DataFrame(rows).sort_values(['event_f1', 'minute_f1'], ascending=False)
        candidates = tuning[tuning.event_f1 >= tuning.event_f1.max() - 0.005]
        selected = candidates.sort_values('minute_f1', ascending=False).iloc[0]
        thresholds = {}
        for house, mode in modes.items():
            span = mode['warm'] - mode['idle']
            thresholds[house] = (mode['idle'] + selected.upper_fraction * span, mode['idle'] + selected.lower_fraction * span)
        held_out = earlier_event_detect_events(grids['H1'].loc[config.midpoint:], *thresholds['H1'])
        labels = reference_minutes.loc[config.midpoint:]
        metrics = earlier_event_score(held_out, test_reference, labels)
        return {'tuning': tuning, 'modes': modes, 'thresholds': thresholds, 'events': held_out, 'reference_events': test_reference, 'reference_minutes': labels, 'metrics': metrics}


In [ ]:
if reproduce_study:
    earlier_event_validate_inputs(EARLIER_INPUTS / "events")
    earlier_temperature_grids, earlier_temperature_reference = earlier_event_prepare_signals(EARLIER_INPUTS / "events")
    earlier_detector = earlier_event_tune_and_score(earlier_temperature_grids, earlier_temperature_reference)
    earlier_reference_minutes = earlier_detector["reference_minutes"]
    earlier_whole_window = pd.DataFrame({
        "start": [earlier_reference_minutes.index.min()],
        "end": [earlier_reference_minutes.index.max()],
        "censored": [True],
        "dur": [(earlier_reference_minutes.index.max() - earlier_reference_minutes.index.min()).total_seconds() / 60],
    })
    earlier_whole_window_scores = earlier_event_score(earlier_whole_window, earlier_detector["reference_events"], earlier_reference_minutes)
    earlier_detector_scores = pd.DataFrame([
        {"comparison": name, "event_f1": scores["F1"], "minute_f1": scores["mF1"], "detected_episodes": scores["n_det"], "reference_episodes": scores["n_gt"]}
        for name, scores in [
            ("Original detector against temperature-derived labels", earlier_detector["metrics"]),
            ("Whole-window overlap control", earlier_whole_window_scores),
        ]
    ])
    earlier_detector_scores.to_csv(EARLIER_OUTPUTS / "temperature_derived_detector_scores.csv", index=False)
    earlier_detector_scores;


In [ ]:
if reproduce_study:
    def earlier_event_load_gas(folder):
        frame = pd.read_parquet(folder / 'h2_gas_intervals.parquet')
        assert list(frame.columns) == ['interval_start', 'interval_end', 'kwh']
        assert not frame.interval_start.duplicated().any()
        assert frame.notna().all().all() and (frame.kwh >= 0).all()
        assert (frame.interval_end - frame.interval_start == pd.Timedelta(minutes=30)).all()
        local = pd.DatetimeIndex(frame.interval_start).tz_localize(earlier_event_CONFIG.timezone, ambiguous='raise', nonexistent='raise')
        frame['day'] = frame.interval_start.dt.strftime('%Y-%m-%d')
        counts = frame.groupby('day').kwh.agg(['sum', 'count'])
        excluded = counts.index.isin(['2026-04-30', '2026-08-11']) | (counts['count'] < 46)
        keep = counts.index[~excluded]
        frame['timestamp'] = local.tz_convert('UTC')
        gas = frame[frame.day.isin(keep)].set_index('timestamp').kwh.sort_index()
        assert len(keep) == 226
        return (gas, counts.loc[excluded])

    def earlier_event_class_minutes(grid, events):
        slope = grid.interpolate(limit=5).diff().rolling(5, center=True, min_periods=1).mean()
        frames = []
        for event_id, event in events.iterrows():
            segment = grid.loc[event.start:event.end].interpolate(limit=5).dropna()
            if segment.empty:
                continue
            derivative = slope.loc[segment.index].to_numpy()
            classes = np.where(derivative >= 0.15, 'rise', np.where(derivative <= -0.15, 'fall', 'hold'))
            frames.append(pd.DataFrame({'timestamp': segment.index, 'event_id': event_id, 'class': classes}))
        minutes = pd.concat(frames, ignore_index=True)
        minutes['bin'] = minutes.timestamp.dt.floor('30min')
        return minutes

    def earlier_event_class_matrix(minutes, gas_index):
        matrix = minutes.groupby(['bin', 'class']).size().unstack(fill_value=0)
        return matrix.reindex(columns=['rise', 'hold', 'fall'], fill_value=0).reindex(gas_index, fill_value=0) / 60

    @dataclass(frozen=True)
    class earlier_event_GasAllocator:
        coefficients: np.ndarray

        @classmethod
        def fit(cls, matrix, gas):
            design = np.c_[np.ones(len(matrix)), matrix.to_numpy()]
            fitted = np.linalg.lstsq(design, gas.to_numpy(), rcond=None)[0]
            return cls(np.maximum(fitted, 0))

        def predict(self, matrix):
            return pd.Series(self.coefficients[0] + matrix.to_numpy() @ self.coefficients[1:], index=matrix.index)

    def earlier_event_later_month_predictions(matrix, gas):
        months = gas.index.tz_convert(earlier_event_CONFIG.timezone).tz_localize(None).to_period('M')
        outputs, folds = ([], [])
        for month in pd.period_range('2026-02', '2026-08', freq='M'):
            training, test = (months < month, months == month)
            if training.sum() < 100 or not test.any():
                continue
            model = earlier_event_GasAllocator.fit(matrix.loc[training], gas.loc[training])
            predicted = model.predict(matrix.loc[test])
            single_matrix = matrix.sum(axis=1).to_frame('event_hours')
            single_model = earlier_event_GasAllocator.fit(single_matrix.loc[training], gas.loc[training])
            training_time = gas.index[training].tz_convert(earlier_event_CONFIG.timezone)
            test_time = gas.index[test].tz_convert(earlier_event_CONFIG.timezone)
            half_hour_training = training_time.hour * 2 + (training_time.minute >= 30).astype(int)
            half_hour_test = test_time.hour * 2 + (test_time.minute >= 30).astype(int)
            climatology = pd.Series(gas.loc[training].to_numpy()).groupby(half_hour_training).mean()
            outputs.append(pd.DataFrame({'meter': gas.loc[test], 'slope_classes': predicted, 'single_power': single_model.predict(single_matrix.loc[test]), 'fixed_power': matrix.loc[test].sum(axis=1) * 16.8, 'persistence': gas.shift(48).reindex(predicted.index).fillna(0), 'time_of_day': climatology.reindex(half_hour_test).to_numpy(), 'fold': str(month)}))
            folds.append({'month': str(month), 'training_bins': int(training.sum()), 'test_bins': int(test.sum()), 'base_kwh_per_bin': model.coefficients[0], **{f'{name}_kw': value for name, value in zip(matrix.columns, model.coefficients[1:])}})
        return (pd.concat(outputs), pd.DataFrame(folds))

    def earlier_event_prediction_metrics(predictions):
        numeric = predictions.drop(columns='fold')
        grains = {'Half-hour': numeric, 'Hour': numeric.resample('1h').sum(), 'Day': numeric.groupby(numeric.index.tz_convert(earlier_event_CONFIG.timezone).date).sum()}
        rows = []
        for grain, frame in grains.items():
            for model in frame.columns.drop('meter'):
                error = frame[model] - frame.meter
                rows.append({'grain': grain, 'model': model, 'n': len(frame), 'correlation': frame[model].corr(frame.meter), 'mae_kwh': error.abs().mean(), 'median_absolute_error_pence': error.abs().median() * 6.31})
        return (pd.DataFrame(rows), grains)


In [ ]:
if reproduce_study:
    earlier_gas, earlier_excluded_gas_days = earlier_event_load_gas(EARLIER_INPUTS / "events")
    earlier_h2_grid = earlier_temperature_grids["H2"]
    earlier_h2_episodes = earlier_event_detect_events(earlier_h2_grid, *earlier_detector["thresholds"]["H2"])
    earlier_h2_minutes = earlier_event_class_minutes(earlier_h2_grid, earlier_h2_episodes)
    earlier_h2_features = earlier_event_class_matrix(earlier_h2_minutes, earlier_gas.index)
    earlier_gas_predictions, earlier_gas_folds = earlier_event_later_month_predictions(earlier_h2_features, earlier_gas)
    earlier_gas_metrics, earlier_gas_grains = earlier_event_prediction_metrics(earlier_gas_predictions)
    earlier_h2_raw_flow = earlier_event_load_temperature(EARLIER_INPUTS / "events", "h2_flow.parquet")
    earlier_gas_months = earlier_gas.index.tz_convert(earlier_event_CONFIG.timezone).tz_localize(None).to_period("M")
    earlier_tuning = earlier_detector["tuning"]
    earlier_selected_fractions = earlier_tuning[earlier_tuning.event_f1 >= earlier_tuning.event_f1.max() - 0.005].sort_values("minute_f1", ascending=False).iloc[0]
    earlier_threshold_predictions = []
    earlier_threshold_periods = []
    for earlier_month in pd.period_range("2026-02", "2026-08", freq="M"):
        earlier_training = earlier_gas_months < earlier_month
        earlier_test = earlier_gas_months == earlier_month
        if earlier_training.sum() < 100 or not earlier_test.any():
            continue
        earlier_cutoff = pd.Timestamp(earlier_month.start_time).tz_localize(earlier_event_CONFIG.timezone).tz_convert("UTC")
        earlier_prefix = earlier_event_grid_1min(earlier_h2_raw_flow.loc[earlier_h2_raw_flow.index < earlier_cutoff])
        earlier_idle, earlier_warm, _, _ = earlier_event_flow_modes(earlier_prefix.dropna().to_numpy())
        earlier_thresholds = (
            earlier_idle + earlier_selected_fractions.upper_fraction * (earlier_warm - earlier_idle),
            earlier_idle + earlier_selected_fractions.lower_fraction * (earlier_warm - earlier_idle),
        )
        earlier_training_episodes = earlier_event_detect_events(earlier_prefix, *earlier_thresholds)
        earlier_training_minutes = earlier_event_class_minutes(earlier_prefix, earlier_training_episodes)
        earlier_training_features = earlier_event_class_matrix(earlier_training_minutes, earlier_gas.index[earlier_training])
        earlier_test_episodes = earlier_event_detect_events(earlier_h2_grid, *earlier_thresholds)
        earlier_test_minutes = earlier_event_class_minutes(earlier_h2_grid, earlier_test_episodes)
        earlier_test_features = earlier_event_class_matrix(earlier_test_minutes, earlier_gas.index[earlier_test])
        earlier_allocator = earlier_event_GasAllocator.fit(earlier_training_features, earlier_gas.loc[earlier_training])
        earlier_threshold_predictions.append(pd.DataFrame({
            "meter": earlier_gas.loc[earlier_test],
            "slope_classes": earlier_allocator.predict(earlier_test_features),
            "fold": str(earlier_month),
        }))
        earlier_threshold_periods.append({"month": str(earlier_month), "opening_temperature_C": earlier_thresholds[0], "closing_temperature_C": earlier_thresholds[1]})
    earlier_threshold_predictions = pd.concat(earlier_threshold_predictions)
    earlier_threshold_metrics, _ = earlier_event_prediction_metrics(earlier_threshold_predictions)
    earlier_hour_counts = earlier_gas_predictions.resample("1h").size()
    earlier_observed_hours = earlier_gas_predictions.drop(columns="fold").resample("1h").sum().loc[earlier_hour_counts > 0]
    earlier_hourly_correction = pd.DataFrame([{
        "empty_hours_removed": int(earlier_hour_counts.eq(0).sum()),
        "observed_hours": len(earlier_observed_hours),
        "original_hourly_mae_kwh": float(earlier_gas_metrics.loc[earlier_gas_metrics.grain.eq("Hour") & earlier_gas_metrics.model.eq("slope_classes"), "mae_kwh"].iloc[0]),
        "observed_hourly_mae_kwh": float((earlier_observed_hours.slope_classes - earlier_observed_hours.meter).abs().mean()),
    }])
    earlier_gas_sensitivity = pd.concat([
        earlier_gas_metrics.loc[earlier_gas_metrics.model.eq("slope_classes")].assign(variant="Full-record temperature cut-offs"),
        earlier_threshold_metrics.assign(variant="Earlier-only cut-offs; later-known episode features"),
    ], ignore_index=True)
    earlier_gas_sensitivity.to_csv(EARLIER_OUTPUTS / "retrospective_gas_threshold_sensitivity.csv", index=False)
    earlier_hourly_correction.to_csv(EARLIER_OUTPUTS / "gas_empty_hour_correction.csv", index=False)
    earlier_gas_predictions.to_csv(EARLIER_OUTPUTS / "retrospective_gas_predictions.csv")
    earlier_threshold_predictions.to_csv(EARLIER_OUTPUTS / "earlier_threshold_gas_predictions.csv")
    earlier_gas_sensitivity;


In [ ]:
if reproduce_study:
    @dataclass(frozen=True)
    class earlier_transfer_SignatureConfig:
        minimum_temperature: float = 10.0
        maximum_temperature: float = 24.0
        temperature_step: float = 0.25
        minimum_weather_hours: int = 20
        minimum_training_days: int = 60
        minimum_test_days: int = 10
        profile_loss_factor: float = 1.05
        seeds: tuple = (42, 1337, 2026)

        @property
        def temperature_grid(self):
            return np.round(np.arange(self.minimum_temperature, self.maximum_temperature + 1e-09, self.temperature_step), 2)

    @dataclass(frozen=True)
    class earlier_transfer_SignatureFit:
        balance_temperature_c: float
        slope_kwh_per_degree_day: float
        base_kwh_per_day: float
        r_squared: float
        days: int
        profile_low_c: float
        profile_high_c: float
        grid_boundary: bool

        def predict(self, degree_days):
            return np.maximum(self.base_kwh_per_day + self.slope_kwh_per_degree_day * degree_days[self.balance_temperature_c].to_numpy(), 0.0)

    class earlier_transfer_DailySignatureModel:

        def __init__(self, config=None):
            self.config = config or earlier_transfer_SignatureConfig()

        def degree_days(self, hourly_temperature):
            temperature = hourly_temperature.dropna()
            days = pd.Index(temperature.index.date, name='date')
            deficits = np.maximum(0.0, self.config.temperature_grid[None, :] - temperature.to_numpy()[:, None])
            matrix = pd.DataFrame(deficits, index=days, columns=self.config.temperature_grid).groupby(level=0).mean()
            counts = temperature.groupby(days).size()
            matrix = matrix.loc[counts[counts >= self.config.minimum_weather_hours].index]
            matrix.index = pd.to_datetime(matrix.index)
            return matrix

        def fit(self, gas, degree_days):
            outcome = np.asarray(gas, dtype=float)
            slopes, bases, losses = earlier_transfer_constrained_grid_fit(outcome, np.asarray(degree_days, dtype=float))
            chosen = int(np.argmin(losses))
            grid = self.config.temperature_grid
            profile = grid[losses <= losses[chosen] * self.config.profile_loss_factor]
            total_variation = np.square(outcome - outcome.mean()).sum()
            return earlier_transfer_SignatureFit(float(grid[chosen]), float(slopes[chosen]), float(bases[chosen]), float(1.0 - losses[chosen] / total_variation) if total_variation > 0 else np.nan, len(outcome), float(profile.min()), float(profile.max()), chosen in (0, len(grid) - 1))

    def earlier_transfer_constrained_grid_fit(outcome, degree_days):
        outcome = np.asarray(outcome, dtype=float)
        degree_days = np.asarray(degree_days, dtype=float)
        temperature_means = degree_days.mean(axis=0)
        outcome_mean = outcome.mean()
        centred_temperature = degree_days - temperature_means
        variance = np.square(centred_temperature).sum(axis=0)
        covariance = (centred_temperature * (outcome - outcome_mean)[:, None]).sum(axis=0)
        slopes = np.where(variance > 0, covariance / np.where(variance > 0, variance, 1.0), 0.0)
        bases = outcome_mean - slopes * temperature_means
        squared_temperature = np.square(degree_days).sum(axis=0)
        origin_slopes = np.maximum((degree_days * outcome[:, None]).sum(axis=0) / np.where(squared_temperature > 0, squared_temperature, 1.0), 0.0)
        negative_base = bases < 0
        slopes[negative_base] = origin_slopes[negative_base]
        bases[negative_base] = 0.0
        negative_slope = slopes < 0
        slopes[negative_slope] = 0.0
        bases[negative_slope] = max(outcome_mean, 0.0)
        residuals = outcome[:, None] - (bases[None, :] + slopes[None, :] * degree_days)
        return (slopes, bases, np.square(residuals).sum(axis=0))


In [ ]:
if reproduce_study:
    def earlier_transfer_load_signature_inputs(folder, model):
        folder = Path(folder)
        manifest = json.loads((folder / 'input_manifest.json').read_text())
        for item in manifest:
            digest = hashlib.sha256((folder / item['file']).read_bytes()).hexdigest()
            if digest != item['sha256']:
                raise ValueError(f"Input changed: {item['file']}")
        missing_h1 = {'2025-12-01', '2026-03-29', '2026-04-21', '2026-04-22'} | {f'2026-02-{day:02d}' for day in range(5, 11)}
        houses = {}
        coverage = []
        for house in ('H1', 'H2'):
            daily = pd.read_csv(folder / f'{house.lower()}_daily_gas.csv', parse_dates=['date'], float_precision='round_trip').set_index('date')
            if house == 'H1':
                eligible = daily.observations.ge(20) & ~daily.index.strftime('%Y-%m-%d').isin(missing_h1)
            else:
                eligible = daily.observations.ge(46) & ~daily.index.strftime('%Y-%m-%d').isin(['2026-04-30', '2026-08-11'])
            hourly = pd.read_csv(folder / f'{house.lower()}_hourly_weather.csv', float_precision='round_trip')
            hourly.index = pd.to_datetime(hourly.timestamp_utc, utc=True).dt.tz_convert('Europe/London')
            temperature = hourly.temperature_c
            degree_days = model.degree_days(temperature)
            common_days = daily.index[eligible].intersection(degree_days.index)
            gas = daily.loc[common_days, 'gas_kwh']
            if gas.isna().any() or gas.lt(0).any() or (not common_days.is_unique):
                raise ValueError(f'Invalid daily observations for {house}')
            houses[house] = {'gas': gas, 'degree_days': degree_days.loc[common_days], 'temperature': temperature, 'weather': 'Local outdoor sensor' if house == 'H1' else 'ERA5 gridded weather'}
            coverage.append({'home': house, 'source_gas_days': len(daily), 'gas_days_after_rules': int(eligible.sum()), 'matched_days': len(common_days), 'first_day': str(common_days.min().date()), 'last_day': str(common_days.max().date()), 'weather': houses[house]['weather']})
        return (houses, pd.DataFrame(coverage))

    def earlier_transfer_fit_houses(houses, model):
        fits = {house: model.fit(values['gas'], values['degree_days']) for house, values in houses.items()}
        summary = pd.DataFrame([{'home': house, 'weather': houses[house]['weather'], **asdict(fit)} for house, fit in fits.items()])
        return (fits, summary)


In [ ]:
if reproduce_study:
    def earlier_transfer_estimates(house_observations, fitted_signatures):
        rows = []
        for donor, recipient in [("H1", "H2"), ("H2", "H1")]:
            observations = house_observations[recipient]
            gas = observations["gas"]
            summer_average = float(gas.loc[gas.index.month.isin([6, 7, 8])].mean())
            donor_fit = fitted_signatures[donor]
            transferred = np.maximum(summer_average + donor_fit.slope_kwh_per_degree_day * observations["degree_days"][donor_fit.balance_temperature_c].to_numpy(), 0)
            local = fitted_signatures[recipient].predict(observations["degree_days"])
            rows.extend({"direction": f"{donor} to {recipient}", "date": str(date.date()), "transferred_kwh": float(transfer), "local_kwh": float(local_estimate)} for date, transfer, local_estimate in zip(gas.index, transferred, local))
        return pd.DataFrame(rows)

    earlier_signature_model = earlier_transfer_DailySignatureModel()
    earlier_houses, earlier_signature_coverage = earlier_transfer_load_signature_inputs(EARLIER_INPUTS / "signatures", earlier_signature_model)
    earlier_full_record_fits, earlier_signature_summary = earlier_transfer_fit_houses(earlier_houses, earlier_signature_model)
    earlier_transfer_predictions = earlier_transfer_estimates(earlier_houses, earlier_full_record_fits)
    earlier_transfer_dependencies = []
    for earlier_donor, earlier_recipient in [("H1", "H2"), ("H2", "H1")]:
        earlier_direction = f"{earlier_donor} to {earlier_recipient}"
        earlier_changed_houses = copy.deepcopy(earlier_houses)
        earlier_summer = earlier_houses[earlier_recipient]["gas"].index.month.isin([6, 7, 8])
        earlier_changed_houses[earlier_recipient]["gas"].loc[earlier_summer] += 10
        earlier_changed_predictions = earlier_transfer_estimates(earlier_changed_houses, earlier_full_record_fits)
        earlier_before = earlier_transfer_predictions.loc[earlier_transfer_predictions.direction.eq(earlier_direction)]
        earlier_after = earlier_changed_predictions.loc[earlier_changed_predictions.direction.eq(earlier_direction)]
        earlier_pre_summer = pd.to_datetime(earlier_before.date).lt("2026-06-01").to_numpy()
        earlier_shift = earlier_after.transferred_kwh.to_numpy() - earlier_before.transferred_kwh.to_numpy()
        assert earlier_houses[earlier_recipient]["gas"].loc[~earlier_summer].equals(earlier_changed_houses[earlier_recipient]["gas"].loc[~earlier_summer])
        assert np.allclose(earlier_shift, 10, atol=1e-12, rtol=0)
        assert np.array_equal(earlier_before.local_kwh, earlier_after.local_kwh)
        earlier_transfer_dependencies.append({
            "direction": earlier_direction,
            "earlier_dates": int(earlier_pre_summer.sum()),
            "summer_gas_increase_kwh_per_day": 10,
            "earlier_estimate_increase_kwh_per_day": float(earlier_shift[earlier_pre_summer].mean()),
            "local_fit_dates": earlier_full_record_fits[earlier_recipient].days,
            "scoring_dates": len(earlier_before),
        })
    earlier_transfer_dependencies = pd.DataFrame(earlier_transfer_dependencies)
    earlier_transfer_dependencies.to_csv(EARLIER_OUTPUTS / "summer_gas_transfer_dependence.csv", index=False)
    earlier_transfer_predictions.to_csv(EARLIER_OUTPUTS / "full_record_transfer_estimates.csv", index=False)
    earlier_transfer_dependencies;


In [ ]:
if reproduce_study:
    HEATING_SCENARIO_REFERENCE_TEMPERATURES = np.round(np.arange(10.0, 24.0 + 1e-09, 0.25), 2)

    def validate_heating_scenario_inputs(folder):
        manifest = json.loads((folder / 'input_manifest.json').read_text())
        needed = {'h1_signature_inputs.parquet', 'h2_signature_inputs.parquet', 'model_parameters.json'}
        inputs = [item for item in manifest['inputs'] if item['input_file'] in needed]
        assert {item['input_file'] for item in inputs} == needed
        for item in inputs:
            actual = hashlib.sha256((folder / item['input_file']).read_bytes()).hexdigest()
            assert actual == item['sha256'], item['input_file']
        return pd.DataFrame([{'input': item['input_file'], 'verified': True} for item in inputs])

    def fit_constrained_heating_profile(gas, degree_days):
        y = np.asarray(gas, float)
        h = np.asarray(degree_days, float)
        hm = h.mean(axis=0)
        ym = y.mean()
        centred = h - hm
        sxx = (centred ** 2).sum(axis=0)
        sxy = (centred * (y - ym)[:, None]).sum(axis=0)
        slope = np.where(sxx > 0, sxy / np.where(sxx > 0, sxx, 1.0), 0.0)
        intercept = ym - slope * hm
        hh = (h ** 2).sum(axis=0)
        origin_slope = np.maximum((h * y[:, None]).sum(axis=0) / np.where(hh > 0, hh, 1.0), 0.0)
        negative_base = intercept < 0
        slope[negative_base] = origin_slope[negative_base]
        intercept[negative_base] = 0.0
        negative_slope = slope < 0
        slope[negative_slope] = 0.0
        intercept[negative_slope] = max(ym, 0.0)
        residual = y[:, None] - (intercept[None, :] + slope[None, :] * h)
        return (slope, intercept, (residual ** 2).sum(axis=0))

    def fit_heating_scenario_profile(frame):
        h = frame.filter(like='hdd_').to_numpy()
        slope, base, loss = fit_constrained_heating_profile(frame.gas_kwh, h)
        best = int(np.argmin(loss))
        return {'T_b': float(HEATING_SCENARIO_REFERENCE_TEMPERATURES[best]), 'm': float(slope[best]), 'E_base': float(base[best])}


In [ ]:
if reproduce_study:
    @dataclass(frozen=True)
    class HeatingScenarioModel:
        house: str
        signature: dict
        emitter: dict
        constants: dict
        pump_differences: dict
        room_reference: float = 20.0

        @property
        def heat_loss_coefficient(self):
            return self.signature['m'] * np.mean(self.constants['eta_band']) / 24

        def achievable_room_temperature(self, outdoor, capped_mean_water):
            k = self.emitter['K']
            exponent = self.constants['en442_n']
            heat_loss = self.heat_loss_coefficient
            balance = lambda room: k * max(capped_mean_water - room, 0.0) ** exponent - heat_loss * (room - outdoor)
            low, high = (outdoor, capped_mean_water)
            if balance(low) < 0:
                return np.nan
            for _ in range(80):
                middle = 0.5 * (low + high)
                if balance(middle) > 0:
                    low = middle
                else:
                    high = middle
            return 0.5 * (low + high)

        def evaluate(self, outdoor, requested_room, shift=1.0):
            room = float(np.clip(requested_room, *self.constants['comfort_clamp_C']))
            sig = self.signature
            effective_base = sig['T_b'] + shift * (room - self.room_reference)
            hdd = max(0.0, effective_base - outdoor)
            demand = self.heat_loss_coefficient * (room - outdoor)
            result = {'house': self.house, 'outside_temperature_C': float(outdoor), 'requested_room_temperature_C': float(requested_room), 'applied_room_temperature_C': room, 'modelled_heat_demand_kW': demand, 'modelled_total_gas_kWh_day': sig['E_base'] + sig['m'] * hdd, 'modelled_space_heating_gas_kWh_day': sig['m'] * hdd}
            if demand <= 0:
                result.update(modelled_flow_setting_C=None, modelled_return_temperature_C=None, assumed_pump_case=min(self.pump_differences), solver_status='INTERIOR', boundary_deficit_kW=0.0, modelled_achievable_room_temperature_C=room)
                return result
            mean_water = room + (demand / self.emitter['K']) ** (1.0 / self.constants['en442_n'])
            choices = {case: {'flow': mean_water + difference / 2, 'return': mean_water - difference / 2} for case, difference in self.pump_differences.items()}
            cap = self.constants['flow_safety_cap_C']
            ceiling = self.constants['condensing_return_C']
            feasible = {case: choice for case, choice in choices.items() if choice['return'] <= ceiling and choice['flow'] <= cap}
            if feasible:
                case = min(feasible, key=lambda candidate: feasible[candidate]['flow'])
                chosen, status = (feasible[case], 'INTERIOR')
                deficit, achievable = (0.0, room)
            else:
                under_cap = {case: choice for case, choice in choices.items() if choice['flow'] <= cap}
                if under_cap:
                    case = min(under_cap, key=lambda candidate: under_cap[candidate]['return'])
                    chosen, status = (under_cap[case], 'BOUNDARY:condensing')
                    deficit, achievable = (0.0, room)
                else:
                    case = min(choices, key=lambda candidate: choices[candidate]['return'])
                    capped_mean = cap - self.pump_differences[case] / 2
                    delivered = self.emitter['K'] * max(capped_mean - room, 0.0) ** self.constants['en442_n']
                    deficit = demand - delivered
                    achievable = self.achievable_room_temperature(outdoor, capped_mean)
                    chosen = {'flow': cap, 'return': cap - self.pump_differences[case]}
                    status = 'BOUNDARY:flow_cap'
            result.update(modelled_flow_setting_C=round(chosen['flow'], 1), modelled_return_temperature_C=round(chosen['return'], 1), assumed_pump_case=case, solver_status=status, boundary_deficit_kW=round(float(deficit), 3), modelled_achievable_room_temperature_C=None if not np.isfinite(achievable) else round(float(achievable), 2))
            return result

    def build_heating_scenario_models(parameters, signatures):
        constants = parameters['constants']
        pump_map = {int(k): v for k, v in parameters['pump_dt_K'].items()}
        emitters = dict(parameters['emitter'])
        loss = signatures['H1']['m'] * np.mean(constants['eta_band']) / 24
        design = loss * (21.0 - constants['design_outdoor_C'])
        band = tuple((factor * design / 50 ** constants['en442_n'] for factor in (1.5, 2.5)))
        emitters['H1'] = dict(K=float(np.sqrt(band[0] * band[1])), band=band, measured=False)
        return {house: HeatingScenarioModel(house, signatures[house], emitters[house], constants, pump_map, parameters['room_reference_C']) for house in ('H1', 'H2')}


In [ ]:
if reproduce_study:
    def summarise_heating_scenarios(models, tariffs):
        rows = []
        for house, model in models.items():
            for outdoor in (-3, 0, 5, 10, 12, 15):
                for room in (18, 19, 20, 21):
                    row = model.evaluate(outdoor, room)
                    row["scenario_id"] = f"SH-{len(rows) + 1:03d}"
                    row["modelled_heat_demand_kW"] = round(row["modelled_heat_demand_kW"], 3)
                    row["modelled_total_cost_GBP_day"] = round(row["modelled_total_gas_kWh_day"] * tariffs[house] / 100, 3)
                    row["modelled_total_gas_kWh_day"] = round(row["modelled_total_gas_kWh_day"], 2)
                    rows.append(row)
        return pd.DataFrame(rows)

    validate_heating_scenario_inputs(EARLIER_INPUTS / "scenarios")
    heating_scenario_parameters = json.loads((EARLIER_INPUTS / "scenarios/model_parameters.json").read_text())
    heating_scenario_inputs = {home: pd.read_parquet(EARLIER_INPUTS / "scenarios" / f"{home.lower()}_signature_inputs.parquet") for home in ("H1", "H2")}
    heating_scenario_profiles = {home: fit_heating_scenario_profile(frame) for home, frame in heating_scenario_inputs.items()}
    heating_scenario_models = build_heating_scenario_models(heating_scenario_parameters, heating_scenario_profiles)
    heating_scenarios = summarise_heating_scenarios(heating_scenario_models, heating_scenario_parameters["tariff_p_per_kwh"])
    assert len(heating_scenarios) == 48 and heating_scenarios.scenario_id.is_unique
    heating_scenarios.to_csv(EARLIER_OUTPUTS / "central_space_heating_scenarios.csv", index=False)
    heating_scenarios;


In [ ]:
import re


def read_new_collection_inputs(analysis_folder, entry, reference_temperatures):
    home = str(entry["home"])
    if not re.fullmatch(r"[A-Za-z][A-Za-z0-9_-]{0,39}", home):
        raise ValueError("Each home needs a short anonymous label.")
    root = Path(analysis_folder).resolve()
    relative = Path(entry["relative_folder"])
    if relative.is_absolute() or ".." in relative.parts:
        raise ValueError("Prepared data must use a relative folder within this collection.")
    folder = (root / relative).resolve()
    if root not in folder.parents:
        raise ValueError("Prepared data must remain inside this collection folder.")
    manifest = json.loads((folder / "input_manifest.json").read_text(encoding="utf-8"))
    names = [item["file"] for item in manifest]
    if len(names) != len(set(names)) or not {"daily_comparison.csv", "hourly_weather.csv", "preparation.json"}.issubset(names):
        raise ValueError("A household input manifest is incomplete or repeats a file.")
    for item in manifest:
        name = Path(item["file"])
        source = folder / name
        if name.is_absolute() or ".." in name.parts or folder not in source.resolve().parents or source.is_symlink():
            raise ValueError("A household input manifest contains an unsafe file path.")
        if hashlib.sha256(source.read_bytes()).hexdigest() != item["sha256"]:
            raise ValueError("A household input changed after preparation.")
    preparation = json.loads((folder / "preparation.json").read_text(encoding="utf-8"))
    if preparation["home"] != home:
        raise ValueError("The household label differs between the collection and prepared inputs.")
    observations = pd.read_csv(folder / "daily_comparison.csv", parse_dates=["date"], float_precision="round_trip").sort_values("date")
    required = {"date", "gas_kwh", "observations", "expected_observations", "eligible", "gas_eligible", "weather_eligible", "mean_temperature_c", "weather_hours"}
    if not required.issubset(observations.columns) or observations.date.isna().any() or observations.date.duplicated().any():
        raise ValueError("Daily observations need unique dates and the preparation coverage fields.")
    if observations.gas_kwh.dropna().lt(0).any() or not np.isfinite(observations.gas_kwh.dropna()).all():
        raise ValueError("Recorded daily gas contains invalid values.")
    if not observations.eligible.isin([True, False]).all():
        raise ValueError("Each daily observation needs a true or false eligibility value.")
    references = np.asarray(reference_temperatures, dtype=float)
    if references.ndim != 1 or not len(references) or not np.isfinite(references).all() or len(np.unique(references)) != len(references) or 15.5 not in references:
        raise ValueError("Reference temperatures must be distinct finite values and include the fixed 15.5 C comparison.")
    columns = [f"hdd_{value:g}" for value in references]
    if not set(columns).issubset(observations.columns):
        raise ValueError("The selected reference temperatures were not included during preparation.")
    paired = observations.loc[observations.eligible.eq(True)].set_index("date")
    degree_days = paired[columns].copy()
    degree_days.columns = references
    if not np.isfinite(degree_days.to_numpy()).all() or not np.isfinite(paired.gas_kwh).all():
        raise ValueError("Eligible paired days contain missing gas or degree-day values.")
    hourly = pd.read_csv(folder / "hourly_weather.csv", float_precision="round_trip")
    hourly.index = pd.to_datetime(hourly.timestamp_utc, utc=True).dt.tz_convert("Europe/London")
    if not hourly.index.is_unique:
        raise ValueError("Prepared hourly weather contains duplicate hours.")
    model_inputs = {"gas": paired.gas_kwh, "degree_days": degree_days, "temperature": hourly.temperature_c, "weather": "Collected outdoor observations"}
    return observations, model_inputs, preparation



In [ ]:
def plot_new_daily_observations(home, observations, output_folder):
    dates = pd.date_range(observations.date.min(), observations.date.max(), freq="D")
    values = observations.set_index("date").reindex(dates)
    figure, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True, constrained_layout=True)
    axes[0].plot(values.index, values.gas_kwh, color="#888888", linewidth=1)
    enough_gas = values.gas_eligible.eq(True) & values.gas_kwh.notna()
    partial_gas = ~values.gas_eligible.eq(True) & values.gas_kwh.notna()
    axes[0].scatter(values.index[enough_gas], values.loc[enough_gas, "gas_kwh"], color="#333333", s=12, label="Gas passes coverage rules")
    axes[0].scatter(values.index[partial_gas], values.loc[partial_gas, "gas_kwh"], facecolors="none", edgecolors="#555555", s=22, label="Gas below coverage rules or excluded")
    axes[0].set_ylabel("Recorded gas (kWh)")
    axes[0].set_title(f"{home}: new observations")
    axes[0].legend(frameon=False, fontsize=8)
    axes[1].plot(values.index, values.mean_temperature_c, color="#555555", linewidth=1.3)
    axes[1].set_ylabel("Observed outdoor mean (°C)")
    axes[1].set_xlabel("Europe/London date")
    for axis in axes:
        axis.grid(axis="y", color="#dddddd", linewidth=0.6)
        axis.spines[["top", "right"]].set_visible(False)
    figure.autofmt_xdate()
    paths = []
    for extension in ("png", "svg"):
        path = Path(output_folder) / f"{home}_new_observations.{extension}"
        figure.savefig(path, dpi=180, bbox_inches="tight")
        paths.append(path.name)
    plt.close(figure)
    return paths


def plot_new_gas_comparison(home, gas, predictions, monthly_results, output_folder):
    figure, axes = plt.subplots(2, 1, figsize=(11, 7), constrained_layout=True)
    styles = (("#222222", "-"), ("#888888", "--"), ("#555555", ":"))
    assessment_dates = pd.to_datetime(predictions.date).sort_values().unique()
    dates = pd.date_range(assessment_dates.min(), assessment_dates.max(), freq="D")
    actual = gas.reindex(dates)
    axes[0].plot(actual.index, actual, color="#aaaaaa", linewidth=2, label="Observed gas")
    for method, (colour, line_style) in zip(COMPARISON_METHODS, styles):
        estimated = predictions.loc[predictions.model.eq(method)].copy()
        estimated["date"] = pd.to_datetime(estimated.date)
        estimated = estimated.set_index("date").predicted_kwh.reindex(dates)
        axes[0].plot(estimated.index, estimated, color=colour, linestyle=line_style, linewidth=1.2, marker=".", markersize=3, label=method)
    axes[0].set_title(f"{home}: new observations and estimates using observed weather")
    axes[0].set_ylabel("Daily gas (kWh)")
    axes[0].legend(frameon=False, fontsize=8)
    axes[0].tick_params(axis="x", labelrotation=30)
    months = sorted(monthly_results.month.unique())
    positions = np.arange(len(months))
    for position, (method, (colour, line_style)) in enumerate(zip(COMPARISON_METHODS, styles)):
        errors = monthly_results.loc[monthly_results.model.eq(method)].set_index("month").reindex(months)
        axes[1].bar(positions + (position - 1) * 0.25, errors.mae_kwh_day, width=0.25, color=colour, edgecolor="white", label=method)
    counts = monthly_results.groupby("month").test_days.first()
    axes[1].set_xticks(positions, [f"{month}\nn={int(counts.loc[month])}" for month in months])
    axes[1].set_ylabel("MAE (kWh/day)")
    axes[1].set_xlabel("Assessment month and number of assessed dates")
    for axis in axes:
        axis.grid(axis="y", color="#dddddd", linewidth=0.6)
        axis.set_axisbelow(True)
        axis.spines[["top", "right"]].set_visible(False)
    paths = []
    for extension in ("png", "svg"):
        path = Path(output_folder) / f"{home}_new_gas_comparison.{extension}"
        figure.savefig(path, dpi=180, bbox_inches="tight")
        paths.append(path.name)
    plt.close(figure)
    return paths



In [ ]:
def analyse_new_collection(analysis_folder):
    root = Path(analysis_folder)
    manifest = json.loads((root / "prepared" / "new_collection_manifest.json").read_text(encoding="utf-8"))
    if manifest.get("profile") != "new_collection":
        raise ValueError("This analysis route requires a new_collection manifest.")
    homes = manifest.get("homes", [])
    if not homes or len({entry["home"] for entry in homes}) != len(homes):
        raise ValueError("The collection needs at least one uniquely labelled household.")
    minimum_training_days = manifest.get("minimum_training_days", 60)
    minimum_test_days = manifest.get("minimum_test_days", 10)
    if not isinstance(minimum_training_days, int) or minimum_training_days < 60 or not isinstance(minimum_test_days, int) or minimum_test_days < 1:
        raise ValueError("Use at least 60 training dates and at least one assessment date per month.")
    references = manifest["reference_temperatures"]
    results_folder = root / "outputs" / "new_collection"
    results_folder.mkdir(parents=True, exist_ok=True)
    summaries, all_monthly_results, all_predictions = [], [], []
    for entry in homes:
        home = entry["home"]
        observations, household_inputs, preparation = read_new_collection_inputs(root, entry, references)
        observations.to_csv(results_folder / f"{home}_new_daily_observations.csv", index=False)
        figures = plot_new_daily_observations(home, observations, results_folder) if not observations.empty else []
        monthly, predictions = compare_daily_estimates({home: household_inputs}, minimum_training_days=minimum_training_days, minimum_assessment_days=minimum_test_days)
        model_status = "assessed" if not monthly.empty else "insufficient_history"
        explanation = "Parameters use earlier dates; each estimate uses the weather observed on its assessed date. This is a retrospective comparison, not an advance forecast."
        if monthly.empty:
            explanation = f"No assessment month has at least {minimum_training_days} earlier eligible dates and {minimum_test_days} eligible assessment dates. Recorded observations remain available; no model scores were calculated."
        else:
            if not np.isfinite(predictions[["actual_kwh", "predicted_kwh"]].to_numpy()).all() or not np.isfinite(monthly[["mae_kwh_day", "rmse_kwh_day"]].to_numpy()).all():
                raise ValueError("The comparison produced a non-finite gas estimate or error.")
            date_sets = predictions.groupby(["home", "month", "model"]).date.agg(lambda values: tuple(sorted(values)))
            if date_sets.groupby(level=[0, 1]).nunique().ne(1).any():
                raise ValueError("The methods were not assessed on identical dates.")
            figures.extend(plot_new_gas_comparison(home, household_inputs["gas"], predictions, monthly, results_folder))
            all_monthly_results.append(monthly)
            all_predictions.append(predictions)
        summaries.append({
            "home": home,
            "model_status": model_status,
            "explanation": explanation,
            "recorded_gas_dates": int(observations.gas_kwh.notna().sum()),
            "eligible_paired_dates": len(household_inputs["gas"]),
            "assessment_months": int(monthly.month.nunique()) if not monthly.empty else 0,
            "assessment_dates": int(predictions.date.nunique()) if not predictions.empty else 0,
            "minimum_training_days": minimum_training_days,
            "minimum_test_days": minimum_test_days,
            "gas_source_type": preparation["gas_source_type"],
            "figures": figures,
        })
    monthly_columns = ["home", "month", "model", "training_days", "test_days", "mae_kwh_day", "rmse_kwh_day", "balance_temperature_c", "tied_candidates", "tied_low_c", "tied_high_c"]
    prediction_columns = ["home", "date", "month", "model", "actual_kwh", "predicted_kwh"]
    monthly_results = pd.concat(all_monthly_results, ignore_index=True) if all_monthly_results else pd.DataFrame(columns=monthly_columns)
    gas_estimates = pd.concat(all_predictions, ignore_index=True) if all_predictions else pd.DataFrame(columns=prediction_columns)
    monthly_results.to_csv(results_folder / "new_monthly_comparison.csv", index=False)
    gas_estimates.to_csv(results_folder / "new_daily_estimates.csv", index=False)
    summary = {"profile": "new_collection", "scope": "New observations; separate from the submitted paper's retained study results.", "interpretation": "Weather is observed on the assessed day. Errors compare methods on identical dates within each month; these are not advance forecasts or estimates of causal savings.", "households": summaries}
    (results_folder / "new_collection_analysis.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return summary, monthly_results, gas_estimates



In [ ]:
if not reproduce_study:
    new_collection_summary, new_monthly_comparison, new_daily_estimates = analyse_new_collection(ANALYSIS_FOLDER)
    for household_result in new_collection_summary["households"]:
        print(household_result["home"] + ": " + household_result["explanation"])
        for figure_name in household_result["figures"]:
            if figure_name.endswith(".png"):
                display(Image(filename=str(ANALYSIS_FOLDER / "outputs" / "new_collection" / figure_name)))
    if not new_monthly_comparison.empty:
        display(HTML(new_monthly_comparison[["home", "month", "model", "training_days", "test_days", "mae_kwh_day"]].round(3).to_html(index=False)))

